# Лабораторная работа. Подготовка, объединение, группировка и статистический анализ таблиц в pandas 3.0

**Цель работы:** освоить полный цикл анализа табличных данных — от сборки аналитической таблицы из нескольких источников до статистики по группам. Сначала данные **объединяются** (`concat`, `merge`, `join`) и **приводятся к нужной форме** (`melt`, `pivot`, `stack`, `unstack`), затем в них **создаются новые признаки** с помощью пользовательских функций (`map`, `apply`, `lambda`), после чего выполняются **группировка и агрегирование** (`groupby`, `agg`, `transform`, `filter`), **статистический анализ по группам** (описательные статистики, корреляции, накопительные и оконные функции) и построение **сводных таблиц** (`pivot_table`, `crosstab`).

**Предварительные знания:** функции и `lambda` в Python; NumPy (`axis`, `np.where`, статистики); `DataFrame`, индексация, `MultiIndex`.

**Версия pandas:** 3.0. Изменения по сравнению с 2.x отмечены в тексте и в таблице раздела 1.

**Данные.** Все наборы входят в состав библиотек `plotly` и `statsmodels`, которые предустановлены в Google Colab, поэтому ничего скачивать и загружать не нужно:
- **Gapminder** (`px.data.gapminder()`) — продолжительность жизни, население и ВВП на душу населения для 142 стран с 1952 по 2007 год с шагом 5 лет (1704 строки; лицензия CC BY). **Сквозной набор работы;**
- **Акции** (`px.data.stocks()`) — нормированные недельные цены шести технологических компаний за 2018–2019 годы, широкий формат;
- **Выборы** (`px.data.election()`) — результаты выборов мэра Монреаля 2013 года по 58 округам, широкий формат;
- **CO₂** (`statsmodels.datasets.co2`) — еженедельные измерения концентрации CO₂ в атмосфере (обсерватория Мауна-Лоа), 1958–2001 годы, 2284 наблюдения **с пропусками**;
- справочники, составленные вручную: членство стран в группах (G7, G20, ОПЕК на 2007 год, БРИКС) и русские названия континентов. В них сохранены реалистичные проблемы: разные написания названий и страны, которых нет в Gapminder.

**Сквозная идея:** в разделах 2–5 из Gapminder и справочников собирается рабочая таблица `panel`, в разделе 6 в неё добавляются признаки, а в разделах 7–12 она анализируется по группам.

**Содержание**

*Часть I. Форма и объединение таблиц*
1. Подготовка данных и особенности pandas 3.0
2. Длинный и широкий форматы: `melt`, `pivot`, `unstack`, `stack`, `wide_to_long`
3. Объединение «стопкой» и «рядом»: `concat`
4. Слияние по ключам: `merge`
5. Соединение по индексу: `join`. Сборка рабочей таблицы

*Часть II. Новые признаки*

6. Пользовательские функции: `map`, `apply`, `lambda`

*Часть III. Группировка и агрегирование*

7. Механизм `groupby` и способы задания групп
8. Агрегирование групп
9. Статистики по группам: центр, разброс, форма, связь
10. Преобразование групп: `transform`, накопительные функции, `filter`, `apply`
11. Оконные функции и временные ряды
12. Интервалы и сводные таблицы

*Завершение*

13. Задания для самостоятельной работы
14. Контрольные вопросы
15. Краткая памятка


### Как работать с ноутбуком

- Ячейки выполняются **строго сверху вниз**: последующие разделы используют переменные предыдущих (`panel`, `blocs_fixed`, `p07`, `cy`, `skeleton` и др.). Если Python сообщает `NameError`, выполните пропущенные ячейки выше.
- Каждый раздел состоит из **теории** (что делает инструмент, какие у него параметры, где возникают типичные ошибки) и **примеров**. В коде все шаги прокомментированы; в комментариях «Ожидаем» указан результат, с которым стоит сверить свой вывод.
- Числа в выводе (размеры таблиц, число строк) используются как **контроль**: после каждой операции полезно проверять `shape` или `len`, чтобы вовремя заметить потерю или размножение строк.
- Задания в разделе 13 выполняются в отдельных ячейках с подсказками; исходные таблицы изменять не нужно.


## 1. Подготовка данных и особенности pandas 3.0

В Google Colab может быть установлена более ранняя версия pandas. Проверьте версию; при необходимости раскомментируйте установку и **перезапустите среду выполнения** (Runtime → Restart session). Если `plotly` или `statsmodels` не найдены, выполните `!pip install plotly statsmodels`.

**Правило работы с этим ноутбуком:** исходные таблицы `gapminder`, `stocks`, `election`, `co2` не изменяются. Новые столбцы создавайте в копиях (`assign`) или в отдельных переменных. Единственное исключение — рабочая таблица `panel`, которая строится в разделах 5–6 как отдельный объект.

In [ ]:
# =====================================================================================
# Установка и импорт библиотек
# =====================================================================================
# В Google Colab может быть установлена pandas версии 2.x. Чтобы получить 3.0, раскомментируйте
# строку с pip ниже, выполните ячейку и перезапустите среду (Runtime -> Restart session).
# !pip install -U "pandas>=3.0"
import time                        # замер времени выполнения (раздел 6)
import numpy as np                 # массивы, np.select, математические функции
import pandas as pd                # главная библиотека работы с таблицами
import plotly.express as px        # из plotly берём встроенные учебные наборы (скачивать ничего не нужно)
import statsmodels.api as sm       # из statsmodels берём ряд CO2

pd.options.display.max_rows = 20   # показывать не более 20 строк (длинные таблицы выводятся сокращённо)
print('pandas', pd.__version__)    # проверка версии: для работы по этому ноутбуку нужна 3.0 или выше

### Что изменилось в pandas 3.0 (и что устарело ещё раньше)

Версия 3.0 удалила много функциональности, объявленной устаревшей в 2.x. Рекомендуется сначала довести код до 2.3 без предупреждений ([заметки о релизе](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html)).

| Тема | В pandas 3.0 | Где |
|---|---|---|
| Строки | Тип `str` по умолчанию (раньше `object`), пропуск — `NaN` | везде |
| Copy-on-Write | Результат любой операции — независимый объект; `copy=` у `merge`, `concat`, `join` не нужен; цепочечное присваивание `df[a][b] = x` не работает | разделы 3–5 |
| Аргументы | Параметры `merge`, `concat`, `join`, `pivot`, `melt` передавайте **по имени** (`how='left'`, `on='key'`) | разделы 2–5 |
| `DataFrame.stack` | Новая реализация: строки с пропусками **не отбрасываются**; параметры `dropna`, `sort` не используются — удаляйте пропуски через `.dropna()` | раздел 2 |
| `concat` | Пустые и полностью пустые таблицы учитываются при определении типов результата | раздел 3 |
| `pd.col()` | Выражения для `assign` вместо `lambda` | раздел 6 |
| `DataFrame.applymap` | Удалён, используйте `DataFrame.map` | раздел 6 |
| Выбор столбцов у `groupby` | Несколько столбцов — только списком: `g[['a', 'b']]` (кортеж не поддерживается ещё с pandas 2.0) | раздел 8 |
| `groupby(axis=1)` | Параметр `axis` у `groupby` удалён; для столбцов используйте `df.columns` и `dtypes` | раздел 7 |
| Функции в `agg`, `transform` | Передавайте строки (`'sum'`, `'max'`), а не `sum`, `np.max` | разделы 8, 10 |
| `observed` | `True` по умолчанию: для категориальных ключей — только встретившиеся категории | разделы 7, 12 |
| `groupby.apply` | Столбцы-ключи не передаются в функцию, `include_groups=True` запрещён; ключ группы — `g.name` | раздел 10 |
| `skipna` | Методы `sum`, `mean`, `median`, `std`, ... у `groupby` принимают `skipna` | раздел 8 |
| Частоты времени | `'YE'`, `'ME'` вместо `'Y'`, `'M'` | разделы 7, 11 |

In [ ]:
# =====================================================================================
# Загрузка данных
# =====================================================================================
gapminder = px.data.gapminder()      # ДЛИННАЯ таблица «страна x год» (1704 строки) — сквозной набор работы
stocks = px.data.stocks()            # ШИРОКАЯ таблица: строка = неделя, по одному столбцу на компанию
stocks['date'] = pd.to_datetime(stocks['date'])   # даты загружены строками -> переводим в тип datetime
election = px.data.election()        # ШИРОКАЯ таблица: строка = округ, по столбцу на кандидата
co2 = sm.datasets.co2.load_pandas().data           # еженедельный ряд CO2; индекс — даты; есть пропуски (NaN)

# Контрольный вывод: размеры таблиц в формате (строк, столбцов) и число пропусков в ряде CO2.
# Ожидаем: gapminder (1704, 8), 142 страны, годы 1952-2007; пропусков в co2 — 59.
print('gapminder:', gapminder.shape, '| стран:', gapminder['country'].nunique(),
      '| годы:', gapminder['year'].min(), '-', gapminder['year'].max())
print('stocks   :', stocks.shape, '| election:', election.shape, '| co2:', co2.shape, '| пропусков co2:', co2['co2'].isna().sum())
gapminder.head(3)                    # первые три строки: быстрый взгляд на названия и типы столбцов

In [ ]:
# =====================================================================================
# Справочники, составленные вручную (данных об этом в Gapminder нет)
# =====================================================================================
# Словарь «группа -> список стран». Ключ — название группы, значение — перечень входящих стран.
# Названия стран записаны по-английски, как в Gapminder, ЗА ИСКЛЮЧЕНИЕМ специально оставленных проблем:
# 'South Korea' (в Gapminder — 'Korea, Rep.'), а Russia, Qatar, United Arab Emirates в Gapminder отсутствуют.
membership = {
    'G7': ['Canada', 'France', 'Germany', 'Italy', 'Japan', 'United Kingdom', 'United States'],
    'G20': ['Argentina', 'Australia', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Indonesia',
            'Italy', 'Japan', 'Mexico', 'Russia', 'Saudi Arabia', 'South Africa', 'South Korea', 'Turkey',
            'United Kingdom', 'United States'],
    'OPEC': ['Algeria', 'Angola', 'Ecuador', 'Indonesia', 'Iran', 'Iraq', 'Kuwait', 'Libya', 'Nigeria',
             'Qatar', 'Saudi Arabia', 'United Arab Emirates', 'Venezuela'],          # состав на 2007 год
    'BRICS': ['Brazil', 'Russia', 'India', 'China', 'South Africa'],
}

# «Разворачиваем» словарь в ДЛИННУЮ таблицу: одна строка = пара (страна, группа).
# Генератор списка проходит по всем группам (b) и по странам каждой группы (c).
# Страна, входящая в две группы (например, Canada), попадает в таблицу дважды.
blocs = pd.DataFrame([(c, b) for b, cs in membership.items() for c in cs], columns=['country', 'bloc'])

# Справочник континентов: английское название (как в Gapminder) -> русское название
continents = pd.DataFrame({
    'region_en': ['Africa', 'Americas', 'Asia', 'Europe', 'Oceania'],
    'continent_ru': ['Африка', 'Америка', 'Азия', 'Европа', 'Океания'],
})
print('blocs:', blocs.shape, '| continents:', continents.shape)   # ожидаем (44, 2) и (5, 2)

# Часть I. Форма и объединение таблиц

## 2. Длинный и широкий форматы: `melt`, `pivot`, `unstack`, `stack`, `wide_to_long`

**Длинный (long) формат:** каждая строка — одно наблюдение, каждая переменная — столбец (например, `country`, `year`, `lifeExp`). Удобен для `groupby`, фильтрации и графиков. Таблица Gapminder — длинная.

**Широкий (wide) формат:** значения одной переменной разнесены по столбцам (например, годы `1952`, `1957`, ...). Удобен для чтения человеком и матричных операций. Таблица `stocks` — широкая.

| Задача | Из столбцов в строки | Из строк в столбцы |
|---|---|---|
| Работа со столбцами таблицы | `melt` | `pivot` / `pivot_table` |
| Работа с уровнями индекса | `stack` | `unstack` |

`pivot` не агрегирует и требует уникальных пар «индекс–столбец»; `pivot_table` умеет агрегировать повторы (`aggfunc`, подробнее — в разделе 12).

### 2.1 `melt`: из широкого формата в длинный

`melt(id_vars, value_vars, var_name, value_name)` оставляет столбцы `id_vars` как есть, а столбцы `value_vars` «складывает» в два новых столбца: имя бывшего столбца (`var_name`) и его значение (`value_name`). Если `value_vars` не указан, преобразуются все столбцы, кроме `id_vars`.


#### Как выбрать формат

Смена формата **не меняет информацию**, а лишь переставляет её: значения из заголовков столбцов переходят в строки (или обратно). Полезно помнить размеры результата:

- `melt` даёт `число_строк x число_value_vars` строк;
- `pivot` даёт `число_уникальных_index x число_уникальных_columns` ячеек (отсутствующие комбинации заполняются `NaN`).

| Нужно | Лучший формат |
|---|---|
| группировать, фильтровать, строить графики по категориям | длинный |
| сравнивать значения между категориями глазами, считать матричные операции, корреляции между столбцами | широкий |
| хранить результаты наблюдений, где число категорий меняется | длинный |


In [ ]:
# melt: ШИРОКИЙ формат -> ДЛИННЫЙ.
# id_vars='date'  — столбец, который остаётся «как есть» (идентификатор строки);
# все остальные столбцы (GOOG, AAPL, ...) «складываются» в два новых:
#   var_name='ticker'  — в него попадают ИМЕНА бывших столбцов;
#   value_name='price' — в него попадают ЗНАЧЕНИЯ из этих столбцов.
stocks_long = stocks.melt(id_vars='date', var_name='ticker', value_name='price')

# Проверка размера: было 105 строк x 7 столбцов, стало 105 x 6 компаний = 630 строк и 3 столбца.
print(stocks.shape, '->', stocks_long.shape)
stocks_long.head(4)                  # первые строки: все недели компании GOOG (потом пойдут AAPL, AMZN, ...)

In [ ]:
# Второй пример melt: голоса за кандидатов лежат в отдельных столбцах (Coderre, Bergeron, Joly).
candidates = ['Coderre', 'Bergeron', 'Joly']
# id_vars — что оставить (округ и победитель в округе); value_vars — какие именно столбцы «растянуть».
votes = election.melt(id_vars=['district', 'winner'], value_vars=candidates,
                      var_name='candidate', value_name='votes')
# Было 58 строк (по округу), стало 58 x 3 = 174 строки (по паре «округ — кандидат»).
print(election.shape, '->', votes.shape)
# Суммарные голоса за каждого кандидата по всему городу (опять же с помощью groupby)
votes.groupby('candidate')['votes'].sum().sort_values(ascending=False)

### 2.2 `pivot`: из длинного формата в широкий

`df.pivot(index=..., columns=..., values=...)` превращает значения столбца `columns` в заголовки столбцов. У оси столбцов остаётся имя (оно видно над заголовками); убрать его можно через `rename_axis(columns=None)`. Если в паре «индекс–столбец» есть повторы, `pivot` выдаёт `ValueError`.


**Как это работает.** Для `pivot` нужны три «роли» столбцов: `index` (что станет строками), `columns` (значения этого столбца станут заголовками) и `values` (откуда брать числа). Схематично:

```
 country  year  lifeExp                 year   1952  1957
 A        1952  30.0                  country
 A        1957  32.0        pivot ->  A        30.0  32.0
 B        1952  40.0                  B        40.0   NaN
```

Если для какой-то пары нет значения (как для `B` в 1957), в ячейке будет `NaN`. Если, наоборот, для пары есть **несколько** значений, `pivot` не знает, какое выбрать, и выдаёт `ValueError`; тогда используют `pivot_table` с функцией агрегирования.


In [ ]:
# pivot: ДЛИННЫЙ формат -> ШИРОКИЙ (обратная операция к melt).
# index='date'      — что станет строками; columns='ticker' — значения этого столбца станут заголовками;
# values='price'    — из какого столбца взять числа для ячеек.
back = (stocks_long.pivot(index='date', columns='ticker', values='price')
        .reindex(columns=stocks.columns[1:])   # pivot упорядочивает столбцы по алфавиту; возвращаем исходный порядок
        .rename_axis(columns=None))            # убираем имя 'ticker' над заголовками столбцов
# Если всё сделано верно, получим ровно исходную таблицу (дата — в индексе). Ожидаем True.
print('pivot вернул исходную таблицу:', back.equals(stocks.set_index('date')))

In [ ]:
# Страны в строках, годы в столбцах: один показатель (lifeExp) в «матричном» виде.
life_wide = gapminder.pivot(index='country', columns='year', values='lifeExp')
print(life_wide.shape)                                    # ожидаем (142, 12): 142 страны x 12 лет
# .loc[строки, столбцы] — выбираем три страны и три года, чтобы вывод поместился на экран
life_wide.loc[['Japan', 'Nigeria', 'Brazil'], [1952, 1982, 2007]]

In [ ]:
# pivot требует, чтобы каждая пара (index, columns) встречалась РОВНО ОДИН раз.
# Здесь пара (континент, год) повторяется для каждой страны континента, поэтому pivot не знает,
# какое из многих значений поставить в ячейку, и выдаёт ValueError.
try:
    gapminder.pivot(index='continent', columns='year', values='lifeExp')
except ValueError as e:
    print('ValueError:', e)      # текст ошибки: «Index contains duplicate entries, cannot reshape»

# pivot_table решает проблему: повторы агрегируются функцией aggfunc (здесь — медиана по странам).
# .round(1) — округляем до одного знака; .loc[:, [...]] — показываем только три года из двенадцати.
gapminder.pivot_table(index='continent', columns='year', values='lifeExp',
                      aggfunc='median').round(1).loc[:, [1952, 1982, 2007]]

In [ ]:
# Если в values передать СПИСОК, получим несколько величин сразу: столбцы становятся двухуровневыми
# (первый уровень — показатель, второй — год).
both = gapminder.pivot(index='country', columns='year', values=['lifeExp', 'gdpPercap'])
print(both.columns.tolist()[:3], '...', both.shape)   # ожидаем (142, 24): 2 показателя x 12 лет
# both['lifeExp'] выбирает верхний уровень -> получаем таблицу только по продолжительности жизни
both['lifeExp'].head(3).loc[:, [1952, 2007]]

### 2.3 `unstack` и `stack`: работа с уровнями индекса

- `unstack(level=-1)` переносит уровень индекса строк в столбцы (длинный → широкий);
- `stack(level=-1)` переносит уровень столбцов в индекс строк (широкий → длинный).

`unstack` создаёт `NaN` в ячейках отсутствующих комбинаций; параметр `fill_value` заменяет их. **В pandas 3.0** `stack` сохраняет строки с `NaN` — чтобы убрать их, вызывайте `.dropna()`.

Отличие от `pivot`: `pivot` работает со **столбцами** обычной таблицы, `unstack` — с **уровнями индекса** (после `set_index` или `groupby`; агрегаты с `unstack` — в разделе 12).


**Уровни индекса.** Индекс может иметь несколько **уровней** (`MultiIndex`), например `continent -> country -> year`. Любой уровень можно указать по имени или номеру. `unstack` берёт один уровень строк и делает из его значений новый уровень **столбцов**; `stack` выполняет обратное действие. Операции взаимно обратны: `df.unstack().stack()` возвращает исходные данные (с точностью до порядка строк и обработки пропусков).

Практическое правило: **`unstack` применяют, когда нужна «матрица» для чтения или расчётов по столбцам** (например, страны x годы); **`stack` — когда нужно вернуть данные в удобный для `groupby` длинный вид**. В `pandas 3.0` после `unstack` пустые ячейки — это `NaN`, и `stack` их сохраняет, поэтому `.dropna()` нужно вызывать явно.


In [ ]:
# Превращаем три столбца в трёхуровневый ИНДЕКС строк: континент -> страна -> год.
# Остаются два числовых столбца (lifeExp, gdpPercap).
idx = gapminder.set_index(['continent', 'country', 'year'])[['lifeExp', 'gdpPercap']]

# unstack('year') переносит уровень индекса 'year' в столбцы: каждому году — свой столбец
# для каждого из двух показателей (получаем двухуровневые столбцы).
w = idx.unstack('year')
print(idx.shape, '->', w.shape, '| имена уровней столбцов:', w.columns.names)   # (1704, 2) -> (142, 24)
w['lifeExp'].loc[:, [1952, 2007]].head(3)     # продолжительность жизни за два года для первых трёх стран

In [ ]:
# stack('year') — обратная операция: годы из столбцов возвращаются в индекс строк.
restored = w.stack('year')
# Сортируем индекс у обеих таблиц (порядок строк мог измениться) и сравниваем. Ожидаем True.
print(restored.shape, '| совпадает с исходной таблицей:', restored.sort_index().equals(idx.sort_index()))

In [ ]:
# Убираем одно наблюдение (Япония, 1952 год), чтобы в «широкой» таблице появилась пустая ячейка.
part = idx.drop(index=('Asia', 'Japan', 1952))       # полный ключ MultiIndex задаётся кортежем
u = part['lifeExp'].unstack('year')                  # широкая таблица «страна x год»: в ней ровно один NaN
print('NaN в таблице:', u.isna().sum().sum())        # .isna() -> True/False; суммируем дважды (по столбцам и итог)

# Возвращаем в длинный вид. В pandas 3.0 stack СОХРАНЯЕТ строки с NaN (всего 142 x 12 = 1704 значений),
# а в 2.x по умолчанию такие строки отбрасывались.
s_all = u.stack()
# Ожидаем в pandas 3.0: 1704 -> 1703 (после dropna исчезает одна строка); в pandas 2.x вывод будет 1703 -> 1703.
print(len(s_all), '->', len(s_all.dropna()), 'после dropna')

### 2.4 `wide_to_long`

Если имена столбцов построены по шаблону «префикс + разделитель + суффикс» (`lifeExp_1952`, `gdpPercap_1952`), удобен `pd.wide_to_long`: он берёт префиксы (`stubnames`) и превращает суффиксы в значения нового столбца.


Параметры: `stubnames` — список префиксов (каждый станет отдельным столбцом значений); `i` — столбец(ы), однозначно идентифицирующий строку исходной таблицы (здесь `country`); `j` — имя нового столбца для суффиксов (здесь `year`); `sep` — разделитель между префиксом и суффиксом (по умолчанию пусто). По умолчанию суффикс должен быть числом (`suffix=r'\d+'`). По сути `wide_to_long` — это `melt` сразу для нескольких групп столбцов, из которых результат собирается в **параллельные** столбцы.


In [ ]:
# wide_to_long работает с именами столбцов вида «префикс_суффикс», например lifeExp_1952.
flat = both.copy()                                    # копия, чтобы не менять таблицу both
# Склеиваем два уровня столбцов в одно имя: ('lifeExp', 1952) -> 'lifeExp_1952'
flat.columns = [f'{v}_{y}' for v, y in flat.columns]
print(flat.shape, flat.columns[:2].tolist())          # ожидаем (142, 24) и ['lifeExp_1952', 'lifeExp_1957']

# stubnames — префиксы (станут столбцами значений); i — столбец-идентификатор строки;
# j — имя нового столбца, куда попадут суффиксы (годы); sep — разделитель между префиксом и суффиксом.
long_back = pd.wide_to_long(flat.reset_index(), stubnames=['lifeExp', 'gdpPercap'],
                            i='country', j='year', sep='_')
# Эталон: исходная длинная таблица с тем же индексом (страна, год), отсортированная.
ref = gapminder.set_index(['country', 'year'])[['lifeExp', 'gdpPercap']].sort_index()
# np.allclose сравнивает числа с допустимой погрешностью. Ожидаем (1704, 2) и True.
print(long_back.shape, '| значения совпадают:', np.allclose(long_back.sort_index(), ref))

## 3. Объединение «стопкой» и «рядом»: `concat`

`pd.concat(objs, axis=0, join='outer', ignore_index=False, keys=None)` склеивает несколько объектов вдоль оси. Он **не ищет совпадения по ключам** — только выравнивает метки другой оси.

| Параметр | Смысл |
|---|---|
| `axis=0` | друг под другом (по умолчанию); выравниваются столбцы |
| `axis=1` | рядом; выравниваются метки индекса |
| `join='outer'` | оставить объединение меток (по умолчанию), недостающее заполняется `NaN` |
| `join='inner'` | оставить только общие метки |
| `ignore_index=True` | выбросить исходный индекс и пронумеровать заново |
| `keys=[...]` | добавить уровень индекса с именем источника (можно передать словарь) |
| `verify_integrity=True` | выдать ошибку, если в индексе результата есть повторы |

Правило: `concat` — когда таблицы имеют **одинаковую структуру** (части одного набора) или общий индекс; `merge`/`join` — когда нужно сопоставить строки по значениям ключа.


**Идея выравнивания.** `concat` не сравнивает значения, а только **выравнивает метки второй оси**: при склейке по строкам выравниваются *столбцы* (по названиям), при склейке по столбцам — *строки* (по индексу). Метки, которых нет в одной из таблиц, дают `NaN` (при `join='outer'`) или отбрасываются (при `join='inner'`).

Что полезно знать:
- индексы при склейке **не перенумеровываются** — одинаковые метки могут повториться (поэтому есть `ignore_index` и `verify_integrity`);
- если в целочисленном столбце появляются `NaN`, его тип меняется на вещественный (`int` -> `float`), что видно в примерах ниже;
- `concat` принимает не только список, но и **словарь**: ключи станут верхним уровнем индекса;
- склеивать `Series` по `axis=1` — удобный способ собрать таблицу из нескольких рядов.


In [ ]:
# Две маленькие таблицы для демонстрации concat. Обратите внимание на ИНДЕКСЫ и СТОЛБЦЫ:
#   df1: индексы 0,1,2;  столбцы a, b, c
#   df2: индексы 2,3,4;  столбцы a, c, d      (индекс 2 есть в обеих таблицах; столбцы b и d — только в одной)
df1 = pd.DataFrame(np.arange(9).reshape(3, 3), index=range(3), columns=['a', 'b', 'c'])
df2 = pd.DataFrame(np.arange(9, 18).reshape(3, 3), index=range(2, 5), columns=['a', 'c', 'd'])
print(df1, df2, sep='\n\n')            # sep='\n\n' — пустая строка между таблицами

In [ ]:
# 1) concat по строкам (axis=0, по умолчанию): таблицы ставятся друг под другом.
#    Столбцы ОБЪЕДИНЯЮТСЯ (join='outer'): недостающие значения заполняются NaN, целые столбцы с пропусками
#    становятся вещественными (b: 1.0, 4.0, 7.0, NaN, ...). sort=True упорядочивает столбцы по алфавиту.
#    Индексы НЕ пересчитываются, поэтому метка 2 встречается дважды.
print(pd.concat([df1, df2], sort=True))

# 2) join='inner' оставляет только общие столбцы (a и c); ignore_index=True нумерует строки заново 0..5.
pd.concat([df1, df2], join='inner', ignore_index=True)

In [ ]:
# concat по столбцам (axis=1): таблицы ставятся рядом, строки выравниваются по ИНДЕКСУ.
# Метки 0 и 1 есть только в df1, метки 3 и 4 — только в df2, метка 2 — в обеих, поэтому остальные ячейки — NaN.
# keys=['left', 'right'] добавляет верхний уровень заголовков, показывающий источник столбцов.
pd.concat([df1, df2], axis=1, keys=['left', 'right'])

In [ ]:
# verify_integrity=True: если в индексе результата есть повторы, concat выбрасывает ошибку.
# Здесь метка 2 есть в обеих таблицах, поэтому ожидаем ValueError «Indexes have overlapping values».
try:
    pd.concat([df1, df2], verify_integrity=True)
except ValueError as e:
    print('ValueError:', e)

In [ ]:
# Реальный сценарий: данные разных периодов хранятся в разных таблицах (частях одного набора).
early = gapminder[gapminder['year'] <= 1977]      # булева маска: строки с 1952 по 1977 год
late = gapminder[gapminder['year'] > 1977]        # строки с 1982 по 2007 год
whole = pd.concat([early, late])                  # склейка по строкам; исходные метки индекса сохраняются
# После сортировки индекса таблица должна совпасть с исходной. Ожидаем 852 + 852 = 1704 и True.
print(len(early), '+', len(late), '=', len(whole), '| совпадает с исходной:', whole.sort_index().equals(gapminder))

In [ ]:
# Словарь частей: ключ — название континента, значение — его подтаблица.
# (Перебор groupby выдаёт пары «значение ключа, подтаблица».)
by_continent = {name: g for name, g in gapminder.groupby('continent')}

# Если передать concat словарь, его ключи становятся ВЕРХНИМ УРОВНЕМ индекса результата.
# names задаёт названия уровней: 'continent' — ключ словаря, 'row' — исходная метка строки.
stacked = pd.concat(by_continent, names=['continent', 'row'])
print(stacked.shape, '| континенты:', stacked.index.get_level_values('continent').unique().tolist())
stacked.loc['Oceania'].head(3)      # по верхнему уровню индекса легко выбрать все строки континента

In [ ]:
# Склейка рядом двух Series -> таблица. Строки выравниваются по индексу (названию страны).
# 1) для каждого года берём срез, ставим страну в индекс и оставляем один столбец; rename задаёт имя столбца
l52 = gapminder.query('year == 1952').set_index('country')['lifeExp'].rename('life_1952')
l07 = gapminder.query('year == 2007').set_index('country')['lifeExp'].rename('life_2007')
# 2) axis=1 — рядом; assign добавляет столбец прироста (разность лет жизни, округлённая до 0.1)
gain = pd.concat([l52, l07], axis=1).assign(gain=lambda d: (d['life_2007'] - d['life_1952']).round(1))
gain.sort_values('gain', ascending=False).head(5)     # пять стран с наибольшим приростом

> **pandas 3.0 и `concat`.** Пустые и целиком пустые таблицы теперь учитываются при определении типа столбцов результата. Если вы собираете таблицу в цикле, заранее отбросьте пустые части: `pd.concat([p for p in parts if not p.empty])`. Не наращивайте таблицу в цикле через `df = pd.concat([df, row])` — это медленно; соберите список частей и вызовите `concat` один раз.

## 4. Слияние по ключам: `merge`

`pd.merge(left, right, how='inner', on=...)` (или `left.merge(right, ...)`) сопоставляет строки двух таблиц по **значениям ключевых столбцов**, как `JOIN` в SQL.

### Типы соединений

| `how` | Результат |
|---|---|
| `'inner'` (по умолчанию) | только ключи, которые есть **в обеих** таблицах |
| `'left'` | все строки левой таблицы; для отсутствующих справа — `NaN` |
| `'right'` | все строки правой таблицы; для отсутствующих слева — `NaN` |
| `'outer'` | все ключи из обеих таблиц; везде, где пары нет, — `NaN` |
| `'cross'` | декартово произведение: каждая строка левой с каждой строкой правой (ключи не нужны) |

Если ключ повторяется в левой таблице $m$ раз, а в правой $n$ раз, он даёт $m \times n$ строк (**many-to-many**), поэтому слияние может неожиданно увеличить число строк.

### Основные аргументы

| Аргумент | Описание |
|---|---|
| `left`, `right` | левая и правая таблицы |
| `how` | тип соединения |
| `on` | столбец или список столбцов-ключей, имеющихся в обеих таблицах |
| `left_on`, `right_on` | ключи, когда имена в таблицах разные |
| `left_index`, `right_index` | использовать индекс как ключ |
| `suffixes` | суффиксы для одноимённых неключевых столбцов (по умолчанию `('_x', '_y')`) |
| `indicator` | добавить столбец `_merge` (`left_only`, `right_only`, `both`) |
| `validate` | проверка типа связи: `'one_to_one'`, `'one_to_many'`, `'many_to_one'`, `'many_to_many'` |
| `sort` | сортировать результат по ключам |

В pandas 3.0 параметр `copy` не нужен, а все аргументы, кроме таблиц, указывайте по имени.

### 4.1 Типы соединений на маленьком примере


### Как работает `merge`

Алгоритм слияния в общих чертах: для каждой строки левой таблицы pandas ищет в правой таблице **все строки с таким же значением ключа** и составляет из них пары; из каждой пары получается одна строка результата (столбцы левой + столбцы правой). Тип соединения `how` определяет, что делать со строками, у которых пары не нашлось.

```
 левая:  key  data1        правая:  key  data2        inner (how='inner'):
         a    0                     a    0                     a  0  0
         b    3                     b    1                     b  3  1
         c    5                     f    3                     b  3  2   (b встречается справа дважды)
```

### Типичные причины неожиданных результатов

| Симптом | Причина | Как проверить |
|---|---|---|
| Строк больше, чем ожидалось | ключ повторяется (one-to-many или many-to-many) | `validate=`, `df['key'].is_unique` |
| Много `NaN` после `left`-слияния | ключи записаны по-разному (регистр, пробелы, `South Korea` / `Korea, Rep.`) | `indicator=True` |
| Строк меньше, чем ожидалось | использован `inner` (по умолчанию), часть ключей не совпала | сравнить `len` до и после |
| Ошибка или пустой результат | разные типы ключей (`int` и `str`) | `df['key'].dtype` |


In [ ]:
# Две маленькие таблицы для разбора типов слияний.
# t1 — левая: ключ 'key' повторяется (a — 3 раза, b — 2 раза), индекс начинается с 3 (нарочно не 0).
# t2 — правая: ключ b встречается 2 раза, есть ключ f, которого нет слева; нет ключей c и d.
t1 = pd.DataFrame({'key': ['a', 'a', 'a', 'b', 'b', 'c', 'd'], 'data1': range(7)}, index=range(3, 10))
t2 = pd.DataFrame({'key': ['a', 'b', 'b', 'f'], 'data2': range(4)})
print(t1, t2, sep='\n\n')

In [ ]:
# Считаем число строк результата для каждого типа соединения:
#   inner: a даёт 3x1=3 строки, b даёт 2x2=4 строки -> 7;
#   left : inner + строки c и d из левой таблицы -> 9;
#   right: inner + строка f из правой таблицы -> 8;
#   outer: inner + c, d + f -> 10.
print({how: len(pd.merge(t1, t2, on='key', how=how)) for how in ['inner', 'left', 'right', 'outer']})

# Сам результат outer. Пустые ячейки (NaN) — там, где у ключа нет пары в другой таблице.
# sort_values упорядочивает строки (порядок строк в результате merge не гарантирован),
# reset_index(drop=True) нумерует их заново.
pd.merge(t1, t2, on='key', how='outer').sort_values(['key', 'data1', 'data2']).reset_index(drop=True)

Порядок строк результата не стоит считать гарантированным: если порядок важен, явно вызывайте `sort_values`.

### 4.2 Ключи, суффиксы, индекс на данных Gapminder

Распространённый приём — «самослияние»: сопоставить таблицу саму с собой, взяв разные годы. Одноимённые неключевые столбцы получают суффиксы.

In [ ]:
# «Самослияние»: берём две выборки одной таблицы (1952 и 2007 годы) и сопоставляем их по стране.
y52 = gapminder.query('year == 1952')[['country', 'continent', 'lifeExp', 'gdpPercap']]
y07 = gapminder.query('year == 2007')[['country', 'lifeExp', 'gdpPercap']]

# В обеих таблицах есть столбцы lifeExp и gdpPercap. suffixes добавляет к ним суффиксы,
# чтобы отличить значения 1952 и 2007 годов: lifeExp_1952, lifeExp_2007 и т. д.
change = y52.merge(y07, on='country', suffixes=('_1952', '_2007'))
change['life_gain'] = (change['lifeExp_2007'] - change['lifeExp_1952']).round(1)    # прирост лет жизни
change['gdp_ratio'] = (change['gdpPercap_2007'] / change['gdpPercap_1952']).round(2)  # во сколько раз вырос ВВП
change.sort_values('life_gain').head(4)       # четыре страны с наименьшим (в том числе отрицательным) приростом

In [ ]:
# Когда ключи называются по-разному, используем left_on (имя в левой таблице) и right_on (имя в правой).
# Цепочка из двух слияний:
#   1) добавляем континент к данным 2007 года (ключ 'country' одинаков в обеих таблицах);
#   2) присоединяем русские названия: столбец continent слева сопоставляется со столбцом region_en справа.
(y07.merge(gapminder[['country', 'continent']].drop_duplicates(), on='country')
    .merge(continents, left_on='continent', right_on='region_en')
    [['country', 'continent', 'continent_ru', 'lifeExp']].head(3))     # оставляем только нужные столбцы

In [ ]:
# Слияние по ДВУМ ключам: строка однозначно определяется парой (страна, год).
# Разрезаем таблицу на две части по столбцам и склеиваем обратно по обоим ключам сразу.
life = gapminder[['country', 'year', 'lifeExp']]
econ = gapminder[['country', 'year', 'gdpPercap', 'pop']]
joined = life.merge(econ, on=['country', 'year'])
# Ожидаем (1704, 5) и True: данные восстановлены полностью.
print(joined.shape, '| совпадает с исходной:', joined.equals(gapminder[['country', 'year', 'lifeExp', 'gdpPercap', 'pop']]))

In [ ]:
# Ключом может быть индекс: left_index=True / right_index=True.
# Ставим страну в индекс обеих таблиц и оставляем по одному столбцу lifeExp.
a = y52.set_index('country')[['lifeExp']]
b = y07.set_index('country')[['lifeExp']]
# Одноимённые столбцы различаем суффиксами.
pd.merge(a, b, left_index=True, right_index=True, suffixes=('_1952', '_2007')).head(3)

### 4.3 Справочник стран по группам: `how='left'`, размножение строк

`blocs` — «длинная» таблица членства: одна строка — пара «страна — группа». Страна может состоять в нескольких группах (Канада — в G7 и G20), поэтому связь «страна → группы» — **one-to-many**. Слияние `left` сохраняет все страны, а число строк вырастает.

**Совет:** используйте `how='left'` от основной таблицы, чтобы не потерять записи, и проверяйте число строк до и после слияния.


**Типы связей между таблицами.** Связь определяется тем, сколько раз ключ встречается в каждой таблице:

| Связь | Ключ слева | Ключ справа | Пример из работы |
|---|---|---|---|
| `one_to_one` | уникален | уникален | страна и её флаг (после `pivot`) |
| `many_to_one` | повторяется | уникален | строки `gapminder` и справочник континентов |
| `one_to_many` | уникален | повторяется | срез за 2007 год (страна один раз) и справочник групп (страна в нескольких группах) |
| `many_to_many` | повторяется | повторяется | почти всегда ошибка: число строк умножается |

Параметр `validate` позволяет **сообщить pandas ожидаемую связь**; если данные ей не соответствуют, возникнет `MergeError`. Это дешёвая защита от ошибок в данных.


In [ ]:
g07 = gapminder.query('year == 2007')       # срез 2007 года: одна строка на страну (142 строки)
# Присоединяем членство в группах. how='left' сохраняет ВСЕ страны, даже без групп (у них bloc = NaN).
# validate='one_to_many': слева каждая страна встречается один раз, справа — может несколько (разные группы);
# если бы это условие нарушилось, pandas выдал бы ошибку MergeError.
m = g07.merge(blocs, on='country', how='left', validate='one_to_many')
# Строк стало больше, чем стран: страна из нескольких групп (Canada: G7 и G20) размножается.
print('строк до:', len(g07), '| после:', len(m))
# Сколько строк в каждой группе. NaN (116) — страны без группы, в том числе Korea, Rep. (имя не совпало!).
m['bloc'].value_counts(dropna=False)

### 4.4 Контроль качества слияния: `indicator` и `validate`

`indicator=True` добавляет столбец `_merge`, показывающий, откуда взята строка. Это основной способ найти записи без пары и сделать **anti-join** (строки одной таблицы, которых нет в другой). `validate` проверяет тип связи и выбрасывает `MergeError`, если данные ему не соответствуют.


**Типичный рабочий цикл при подготовке данных:** (1) слить таблицы с `indicator=True`; (2) посмотреть, сколько строк `left_only` / `right_only`; (3) выяснить причину (опечатка, разный формат записи, отсутствие в справочнике); (4) исправить данные (`replace`, `str.strip()`, `str.lower()`); (5) слить заново и проверить, что нерешённым осталось только то, что действительно отсутствует.


In [ ]:
# Ищем элементы справочника, которым не нашлось пары в Gapminder. Направление важно: слева — blocs.
# indicator=True добавляет столбец _merge со значениями:
#   'both' — пара найдена, 'left_only' — только в левой таблице, 'right_only' — только в правой.
chk = blocs.merge(g07[['country']], on='country', how='left', indicator=True)
print(chk['_merge'].value_counts().to_dict())          # right_only при how='left' быть не может
# Выводим строки без пары: это и есть anti-join (элементы левой таблицы, которых нет в правой).
chk.query('_merge == "left_only"')[['country', 'bloc']]

Среди несовпавших есть `South Korea` — в Gapminder эта страна записана как `Korea, Rep.`. Это ошибка согласования названий, её исправляют заменой. Остальные (`Russia`, `Qatar`, `United Arab Emirates`) в Gapminder действительно отсутствуют.

In [ ]:
# Исправляем расхождение в написании: заменяем 'South Korea' на 'Korea, Rep.' (как в Gapminder).
# replace принимает словарь «старое -> новое»; остальные значения не меняются.
blocs_fixed = blocs.assign(country=blocs['country'].replace({'South Korea': 'Korea, Rep.'}))
chk2 = blocs_fixed.merge(g07[['country']], on='country', how='left', indicator=True)
# Ожидаем три страны, отсутствующих в Gapminder: Qatar, Russia, United Arab Emirates.
print('после исправления без пары:', sorted(chk2.loc[chk2['_merge'] == 'left_only', 'country'].unique()))

# Второй способ anti-join: ~isin(...) выбирает строки Gapminder, чьей страны НЕТ в справочнике групп.
no_bloc = g07[~g07['country'].isin(blocs_fixed['country'])]
print(len(no_bloc), 'стран вне групп, например:', no_bloc['country'].head(5).tolist())

In [ ]:
# validate проверяет ожидаемый тип связи. Здесь заведомо неверное ожидание one_to_one:
# в blocs_fixed страна может повторяться (несколько групп), поэтому pandas выбросит MergeError.
# Так validate защищает от тихого размножения строк.
try:
    g07.merge(blocs_fixed, on='country', validate='one_to_one')
except pd.errors.MergeError as e:
    print('MergeError:', e)

### 4.5 Особенности: размножение строк, `NaN` в ключах, `cross`

- Дубликаты ключей в обеих таблицах дают $m \times n$ строк. Проверяйте уникальность ключей справочников: `continents['region_en'].is_unique`.
- В pandas значения `NaN` в ключах **сопоставляются друг с другом** (в SQL `NULL` с `NULL` не совпадают). Удаляйте или заполняйте такие ключи заранее.
- Типы ключей должны совпадать (`int` и `str` не сопоставляются).
- `how='cross'` полезен для построения «каркаса» всех комбинаций, к которому затем присоединяют факты (полный пример — в разделе 12).

In [ ]:
# 1) many-to-many: ключ 'x' встречается 2 раза слева и 3 раза справа -> результат 2 x 3 = 6 строк.
dup = pd.DataFrame({'k': ['x', 'x'], 'v1': [1, 2]})
dup2 = pd.DataFrame({'k': ['x', 'x', 'x'], 'v2': [10, 20, 30]})
print('many-to-many:', len(dup.merge(dup2, on='k')), 'строк (2 x 3)')

# 2) NaN в ключах СОПОСТАВЛЯЮТСЯ друг с другом (в SQL NULL с NULL не совпадают): получаем 2 строки.
n1 = pd.DataFrame({'k': ['x', None], 'v1': [1, 2]})
n2 = pd.DataFrame({'k': ['x', None], 'v2': [3, 4]})
print('NaN-ключи совпадают:', len(n1.merge(n2, on='k')), 'строки')

# 3) cross: все сочетания «группа x год» без ключей. 4 группы x 2 года = 8 строк.
#    Этот «каркас» пригодится в разделе 12.
skeleton = pd.DataFrame({'bloc': list(membership)}).merge(pd.DataFrame({'year': [1952, 2007]}), how='cross')
print('cross: 4 группы x 2 года =', len(skeleton), 'строк')

## 5. Соединение по индексу: `join`. Сборка рабочей таблицы

`left.join(other, on=None, how='left', lsuffix='', rsuffix='')` — удобная форма `merge` для случая, когда ключом служит **индекс** правой таблицы. Отличия от `merge`:
- по умолчанию `how='left'` (у `merge` — `'inner'`);
- ключ правой таблицы — всегда её индекс; ключом левой может быть индекс (по умолчанию) или столбец (параметр `on`);
- при одноимённых столбцах нужно указать `lsuffix` / `rsuffix`, иначе будет ошибка;
- можно передать список таблиц и соединить сразу несколько по индексу.

| Инструмент | Когда использовать |
|---|---|
| `concat` | таблицы одной структуры (по строкам) или с общим индексом (рядом); ключей-значений нет |
| `merge` | сопоставление по значениям столбцов-ключей; разные имена ключей; контроль через `indicator`, `validate` |
| `join` | сопоставление по индексу; быстро присоединить агрегаты и справочники |


**Как работает `join`.** `left.join(right)` эквивалентно `left.merge(right, left_index=True, right_index=True, how='left')`. Если у левой таблицы ключ — столбец, а не индекс, указывают `on='имя_столбца'`: тогда этот столбец сопоставляется с **индексом правой таблицы**. Поэтому перед `join` справочник удобно подготовить так, чтобы ключ стоял в его индексе (`set_index`). Для присоединения нескольких таблиц с общим индексом можно передать список.


In [ ]:
# Справочник стран: по одной строке на страну, название страны — в индексе.
# drop_duplicates() убирает повторы (в Gapminder каждая страна записана 12 раз — по числу лет).
country_info = gapminder[['country', 'continent', 'iso_alpha']].drop_duplicates().set_index('country')

# Сводка по странам, собранная из широких таблиц (раздел 2): продолжительность жизни в 1952 и 2007 годах
# и население в 2007 году. Индексы этих столбцов — названия стран, поэтому их удобно соединять через join.
pop_wide = gapminder.pivot(index='country', columns='year', values='pop')
stats = pd.DataFrame({'life_1952': life_wide[1952], 'life_2007': life_wide[2007], 'pop_2007': pop_wide[2007]})

# join по индексу, how='left' по умолчанию: берём все страны из country_info и присоединяем к ним stats.
country_info.join(stats).head(3)

In [ ]:
# Сколько групп у каждой страны: value_counts() считает, сколько раз название встречается в справочнике.
# rename('n_blocs') задаёт имя Series, оно станет именем столбца при join.
n_blocs = blocs_fixed['country'].value_counts().rename('n_blocs')

left_res = country_info.join(n_blocs)         # left: сохраняются все 142 страны, у стран без групп — NaN
print('left :', left_res.shape[0], 'стран | без групп:', left_res['n_blocs'].isna().sum())
# inner оставляет только страны, имеющиеся в обеих таблицах (состоящие хотя бы в одной группе). Ожидаем 27.
print('inner:', country_info.join(n_blocs, how='inner').shape[0], 'стран')
# Для читаемости заменяем NaN нулём и приводим к целому; выводим страны с наибольшим числом групп.
left_res.assign(n_blocs=left_res['n_blocs'].fillna(0).astype(int)).sort_values('n_blocs', ascending=False).head(4)

In [ ]:
# Параметр on='country': ключом слева служит СТОЛБЕЦ country таблицы blocs_fixed,
# а справа — ИНДЕКС таблицы country_info (там страна стоит в индексе).
# query оставляет только членов БРИКС. Россия присутствует в справочнике, но её нет в Gapminder — для неё NaN.
blocs_fixed.join(country_info, on='country').query('bloc == "BRICS"')

In [ ]:
# Одноимённые столбцы (lifeExp, pop есть в обеих таблицах) требуют суффиксов: lsuffix для левой, rsuffix для правой.
s52 = gapminder.query('year == 1952').set_index('country')[['lifeExp', 'pop']]
s07 = gapminder.query('year == 2007').set_index('country')[['lifeExp', 'pop']]
print(s52.join(s07, lsuffix='_1952', rsuffix='_2007').head(3))
# join умеет принимать СПИСОК таблиц: присоединяем сразу stats и n_blocs.
country_info.join([stats, n_blocs]).head(3)

### Сборка рабочей таблицы `panel`

Теперь соберём всё вместе. Членство в группах приведём к «широкому» виду: для каждой страны — по одному столбцу-флагу на группу (`pivot` по уникальным парам «страна — группа»). Затем присоединим к Gapminder русские названия континентов (`merge` many-to-one) и флаги (`merge` по индексу правой таблицы). Число строк должно остаться равным 1704.


Этапы сборки (все они используют инструменты разделов 2–5):
1. **`pivot`** превращает длинный список «страна — группа» в таблицу флагов «страна x группа».
2. **`merge` many-to-one** присоединяет русские названия континентов; `validate` гарантирует, что справочник не размножит строки.
3. **`merge` по индексу правой таблицы** присоединяет флаги; `how='left'` сохраняет все страны.
4. **`fillna(0)`** заменяет пропуски у стран без групп нулями (флаг «не состоит»).
5. **Сумма флагов** по строке (`sum(axis=1)`) даёт число групп страны.

После каждого шага полезно проверять число строк: оно должно остаться равным 1704.


In [ ]:
# =====================================================================================
# Сборка рабочей таблицы panel
# =====================================================================================
# Шаг 1. Членство в группах -> «широкий» вид: по одному столбцу-флагу (0/1) на группу.
#   assign(member=1)        — вспомогательный столбец из единиц;
#   pivot(...)              — страны в строки, группы в столбцы; в ячейке 1, если страна состоит в группе, иначе NaN
#                             (pivot применим: пара «страна — группа» уникальна);
#   notna().astype(int)     — NaN -> 0, число -> 1;
#   add_prefix('in_')       — названия флагов: in_G7, in_G20, in_OPEC, in_BRICS;
#   rename_axis(columns=None) — убираем лишнее имя над столбцами.
flags = (blocs_fixed.assign(member=1)
         .pivot(index='country', columns='bloc', values='member')
         .notna().astype(int)
         .add_prefix('in_')
         .rename_axis(columns=None))
flag_cols = flags.columns.tolist()          # список названий флагов пригодится позже

# Шаги 2-3. Присоединяем к Gapminder справочники.
#   1-е слияние: русские названия континентов (many-to-many нет: каждый континент в справочнике один раз);
#   drop(columns='region_en') — ключ справа больше не нужен;
#   2-е слияние: флаги групп; ключ слева — столбец country, справа — индекс таблицы flags;
#   reset_index(drop=True) — нумеруем строки заново.
panel = (gapminder
         .merge(continents, left_on='continent', right_on='region_en', how='left', validate='many_to_one')
         .drop(columns='region_en')
         .merge(flags, left_on='country', right_index=True, how='left')
         .reset_index(drop=True))

# Шаг 4. У стран вне групп флаги пустые (NaN) — заменяем нулями и возвращаем целый тип.
panel[flag_cols] = panel[flag_cols].fillna(0).astype(int)
# Шаг 5. Число групп у страны = сумма флагов по строке (axis=1 — суммируем по столбцам).
panel['n_blocs'] = panel[flag_cols].sum(axis=1)

# Контроль: число строк не изменилось (1704), добавились столбцы continent_ru, in_*, n_blocs.
print('строк:', len(panel), '(было', len(gapminder), ') | столбцы:', panel.columns.tolist())
panel.query('country in ["Canada", "Nigeria", "Japan"] and year == 2007')    # три страны для проверки флагов

# Часть II. Новые признаки

## 6. Пользовательские функции: `map`, `apply`, `lambda`

Функция в Python — объект, который можно передать другой функции. Именно так работают `map` и `apply`; те же функции потом используются в `agg`, `transform` и `groupby.apply`.

| Метод | Что получает функция | Результат |
|---|---|---|
| `Series.map(arg)` | элемент; `arg` — словарь, функция или `Series` | `Series` |
| `Series.apply(func, args=...)` | элемент (допускает доп. аргументы) | `Series` |
| `DataFrame.apply(func, axis=0)` | столбец целиком | по значению на столбец |
| `DataFrame.apply(func, axis=1)` | строка целиком | по значению на строку |
| `DataFrame.map(func)` | каждую ячейку | `DataFrame` (раньше `applymap`) |

`na_action='ignore'` в `map` не передаёт `NaN` в функцию. В своих функциях проверяйте пропуски через `pd.isna`: сравнение с `NaN` всегда `False`.


**`lambda`** — короткая анонимная функция: `lambda x: x * 2` эквивалентна `def f(x): return x * 2`. Её удобно использовать для небольших одноразовых преобразований; для сложной логики (несколько условий, проверки) пишут обычную функцию `def`.

**Что выбрать:**
- нужно *заменить* значения по таблице соответствий -> `map` со словарём или `Series`;
- нужно применить *правило* к каждому значению -> `Series.apply` / `map` с функцией;
- нужна функция, использующая *несколько* столбцов строки -> `DataFrame.apply(axis=1)`;
- нужна сводная характеристика *каждого столбца* -> `DataFrame.apply(axis=0)`.

**Как вызываются функции.** Функция в `map`/`apply` вызывается **по одному разу для каждого объекта** (элемента, столбца или строки), а результаты собираются в новый объект. Именно такая «поэлементная» логика замедляет `apply` на больших таблицах (см. ниже).


In [ ]:
# 1) map со СЛОВАРЁМ: каждое значение заменяется по ключу; ключ, которого нет в словаре, даёт NaN.
#    В словаре нарочно нет 'Oceania', поэтому 24 строки Океании (2 страны x 12 лет) получат NaN.
names = panel['continent'].map({'Africa': 'Африка', 'Americas': 'Америка', 'Asia': 'Азия', 'Europe': 'Европа'})
print('Океания без перевода -> NaN:', names.isna().sum(), 'строк (', panel.query('continent == "Oceania"')['country'].nunique(), 'страны )')

# 2) map с ФУНКЦИЕЙ и аргумент na_action. Маленький ряд с пропуском посередине:
s = pd.Series([315.2, np.nan, 316.1])
# Без na_action функция получает и NaN -> форматирует его в строку 'nan ppm'.
print(s.map(lambda x: f'{x:.1f} ppm').tolist())
# С na_action='ignore' пропуск в функцию НЕ передаётся и остаётся пропуском.
print(s.map(lambda x: f'{x:.1f} ppm', na_action='ignore').tolist())

In [ ]:
# map с Series: значения элементов используются как МЕТКИ индекса другого Series.
# Строим соответствие «страна -> её первая группа в справочнике» (drop_duplicates оставляет первую запись страны).
first_bloc = blocs_fixed.drop_duplicates('country').set_index('country')['bloc']
# Каждой строке panel ставим первую группу её страны; для стран без групп получится NaN.
primary_bloc = panel['country'].map(first_bloc)
# Ожидаем 1380 строк без группы (115 стран x 12 лет) из 1704.
print('строк без группы (страны вне G7, G20, ОПЕК, БРИКС):', primary_bloc.isna().sum(), 'из', len(panel))
primary_bloc.value_counts()         # частоты по группам (NaN не показываются)

In [ ]:
# Пользовательская функция для apply: классифицирует продолжительность жизни.
# Проверка pd.isna(x) в начале ОБЯЗАТЕЛЬНА: сравнение x < 50 для NaN всегда даёт False,
# и без проверки пропуск попал бы в класс 'very_high'.
def life_class(x):
    if pd.isna(x):
        return 'unknown'
    if x < 50:
        return 'low'
    if x < 65:
        return 'medium'
    if x < 75:
        return 'high'
    return 'very_high'

# Функция с дополнительными аргументами: threshold обязателен, label — по умолчанию 'yes'.
def above(value, threshold, label='yes'):
    return label if value > threshold else 'no'

# apply вызывает функцию для каждого элемента столбца; value_counts считает, сколько раз встретился каждый класс.
print(panel['lifeExp'].apply(life_class).value_counts().to_dict())
# Дополнительные аргументы передаются через args=(...) (позиционные) и именованные параметры (label=...).
print(panel['gdpPercap'].apply(above, args=(10000,), label='rich').value_counts().to_dict())

In [ ]:
measures = panel[['lifeExp', 'pop', 'gdpPercap']]
# DataFrame.apply с axis=0 (по умолчанию): функция получает КАЖДЫЙ СТОЛБЕЦ как Series и возвращает число.
# Здесь — размах (максимум минус минимум) каждого столбца.
print(measures.apply(lambda col: col.max() - col.min()).round(1).to_dict())

# axis=1: функция получает КАЖДУЮ СТРОКУ как Series (доступ к полям — по названию столбца).
# Считаем общий ВВП страны = ВВП на душу x население, с проверкой на пропуски.
def gdp_total(row):
    if pd.isna(row['gdpPercap']) or pd.isna(row['pop']):
        return np.nan
    return row['gdpPercap'] * row['pop']

panel.head(3).apply(gdp_total, axis=1)     # три первые строки; результат — Series той же длины

In [ ]:
# DataFrame.map применяет функцию к КАЖДОЙ ЯЧЕЙКЕ (замена удалённого applymap).
# Форматируем числа: тысячи разделяются запятой, один знак после точки. Результат — строки.
panel[['lifeExp', 'gdpPercap']].head(3).map(lambda x: f'{x:,.1f}')

In [ ]:
# pandas >= 3.0: pd.col('имя') ссылается на столбец «в момент вычисления» — это замена lambda в assign.
# В той же ячейке считаем одно и то же двумя способами и сравниваем результаты.
t_demo = panel.assign(gdp_lambda=lambda d: d['gdpPercap'] * d['pop'],             # классический способ
                      gdp_col=pd.col('gdpPercap') * pd.col('pop'))                # новый способ pd.col
print('Результаты совпадают:', np.allclose(t_demo['gdp_lambda'], t_demo['gdp_col']))   # ожидаем True

### Векторизация против `apply`

`apply` вызывает функцию Python для каждого элемента или строки — это цикл «под капотом». Векторные операции pandas/NumPy обрабатывают столбец целиком и быстрее на порядки. **Сначала ищите векторное решение:** арифметику над столбцами, `np.where`, `np.select([усл1, усл2], [знач1, знач2], default)`, `pd.cut`, групповые `transform`. `apply` оставляйте для логики, которую трудно выразить иначе.


**Почему векторные операции быстрее.** Методы pandas и NumPy выполняют цикл по данным внутри оптимизированного кода на C над непрерывным блоком памяти, а `apply` на каждом шаге вызывает функцию Python, создавая объекты (для `axis=1` — целую `Series` на каждую строку). Правило: если задачу можно записать арифметикой, сравнением, `np.where`/`np.select`, `pd.cut` или встроенным методом, используйте их; `apply` оставляйте для сложной логики или небольших таблиц.


In [ ]:
# Сравнение скорости. Для заметной разницы увеличиваем таблицу в 60 раз (~100 000 строк).
big = pd.concat([panel] * 60, ignore_index=True)

# Классификация двумя способами: apply (вызов Python-функции для каждой строки) и np.select (векторно).
# time.perf_counter() — точные часы; разность двух замеров — время выполнения операции.
t = time.perf_counter(); r1 = big['lifeExp'].apply(life_class); t_apply = time.perf_counter() - t
t = time.perf_counter()
# np.select: условия проверяются по порядку, для каждой строки берётся значение, соответствующее
# ПЕРВОМУ выполненному условию; если ни одно не выполнено — default.
r2 = np.select([big['lifeExp'] < 50, big['lifeExp'] < 65, big['lifeExp'] < 75],
               ['low', 'medium', 'high'], default='very_high')
t_vec = time.perf_counter() - t
print('Совпадают:', (r1.to_numpy() == r2).all(), f'| apply: {t_apply:.4f} c | np.select: {t_vec:.4f} c')

# Построчный apply (axis=1) на 20 000 строках против обычного умножения столбцов. Разница — на порядки.
part = big.head(20000)
t = time.perf_counter(); a = part.apply(gdp_total, axis=1); t_row = time.perf_counter() - t
t = time.perf_counter(); b = part['gdpPercap'] * part['pop']; t_v = time.perf_counter() - t
print(f'apply axis=1: {t_row:.4f} c | векторно: {t_v:.4f} c')

### Фиксируем рабочую таблицу

Добавим в `panel` признаки, созданные в этом разделе: основную группу страны и класс продолжительности жизни (векторным способом). Со следующего раздела работаем именно с этой таблицей.

In [ ]:
# Добавляем признаки к рабочей таблице (assign создаёт новую таблицу с дополнительными столбцами).
#   primary_bloc — основная группа страны (вычислена map в одной из предыдущих ячеек);
#   life_class   — класс продолжительности жизни, посчитанный векторно (np.select).
panel = panel.assign(
    primary_bloc=primary_bloc,
    life_class=np.select([panel['lifeExp'] < 50, panel['lifeExp'] < 65, panel['lifeExp'] < 75],
                         ['low', 'medium', 'high'], default='very_high'),
)
print(panel.shape)                                  # ожидаем (1704, 16): 14 прежних столбцов + 2 новых
panel.query('year == 2007').head(3)                 # проверяем результат на данных 2007 года

# Часть III. Группировка и агрегирование

## 7. Механизм `groupby` и способы задания групп

Группировка выполняется в три шага (**split–apply–combine**):
1. **Split** — таблица делится на группы по значениям ключа;
2. **Apply** — к каждой группе применяется операция: *агрегирование* (одно значение на группу), *преобразование* (результат той же формы, что группа) или *отбор* (группа остаётся или исчезает);
3. **Combine** — результаты собираются в один объект.

`df.groupby(key)` ничего не вычисляет: он возвращает «ленивый» объект, который помнит, какие строки в какую группу попадут. Вычисление запускается вызовом метода (`mean()`, `agg()`, `transform()`, ...), поэтому один объект группировки можно использовать несколько раз.

| Член объекта группировки | Что делает |
|---|---|
| `ngroups`, `size()` | число групп; число строк в каждой группе |
| `groups`, `get_group(key)` | метки строк групп; данные одной группы |
| `nth(k)`, `head(n)` | k-я строка и первые n строк каждой группы |
| `cumcount()`, `ngroup()` | номер строки в группе; номер группы |
| `for key, g in groups` | обход пар «ключ — таблица группы» |


**Пример.** Для таблицы из шести строк с ключом `x` (значения `a`, `a`, `b`, `b`, `c`, `c`) `groupby('x')` разобьёт её на три подтаблицы по две строки. Вызов `mean()` посчитает среднее `y` в каждой подтаблице и соберёт три числа в один `Series` с индексом `a`, `b`, `c`. По умолчанию группы **сортируются по ключу**, а строки с `NaN` в ключе **исключаются** (оба поведения настраиваются параметрами `sort` и `dropna`).

Заметьте, что операции `groupby` обычно возвращают **не исходную таблицу, а новую** — меньшую (агрегат), такой же формы (преобразование) или подмножество строк (отбор). От этого зависит, какой метод выбрать (см. таблицу в разделе 10).


In [ ]:
# Маленькая учебная таблица: ключ x (три значения по две строки) и число y.
df = pd.DataFrame({'x': ['a', 'a', 'b', 'b', 'c', 'c'],
                   'y': [2, 4, 0, 5, 5, 10]})
groups = df.groupby('x')              # объект группировки: ничего не вычислено, лишь запомнено разбиение
print('групп:', groups.ngroups, '| размеры:', groups.size().to_dict())      # 3 группы по 2 строки
# groups.groups — словарь «ключ -> метки строк группы» (показывает, как таблица разбита).
print('метки строк:', {k: list(v) for k, v in groups.groups.items()})
print(groups.get_group('b'))          # таблица одной группы

In [ ]:
# cumcount() — порядковый номер строки внутри группы (с нуля); ngroup() — номер самой группы.
print(df.assign(pos_in_group=groups.cumcount(), group_id=groups.ngroup()))
# nth(0) — первая строка каждой группы (метки исходных строк сохраняются).
print(groups.nth(0))

# Три шага split-apply-combine вручную: for key, g in groups перебирает пары «ключ, таблица группы».
# Среднее по каждой группе собираем в словарь и сравниваем с готовым groupby.mean().
manual = {key: g['y'].mean() for key, g in groups}
print(manual, '|', groups['y'].mean().to_dict())      # результаты должны совпасть

In [ ]:
# То же на рабочей таблице: делим 1704 строки на 5 групп по континентам.
by_cont = panel.groupby('continent')
print('групп:', by_cont.ngroups, '| строк:', by_cont.size().to_dict())     # Africa 624, Americas 300, ...
# get_group возвращает подтаблицу одной группы; выбираем три столбца и первые три строки.
by_cont.get_group('Oceania')[['country', 'year', 'lifeExp']].head(3)

### Способы задания групп

| Ключ | Пример |
|---|---|
| столбец или список столбцов | `groupby('continent')`, `groupby(['continent', 'year'])` |
| уровень индекса | `groupby(level='year')` |
| функция (применяется к меткам индекса) | `groupby(len)` |
| массив или `Series` той же длины, в том числе булев | `groupby(df['x'] > 0)` |
| `pd.Grouper` для дат | `groupby(pd.Grouper(freq='YE'))` |


Когда какой способ нужен:
- **столбцы** — основной вариант, когда категории записаны в данных;
- **уровень индекса** — когда категории уже находятся в `MultiIndex` (например, после `set_index` или `unstack`);
- **функция** — когда ключ вычисляется по метке (по длине названия, по первой букве, по первой цифре кода);
- **массив / `Series`** — когда ключ вычисляется из значений (условие «выше медианы») или задан внешним массивом (случайное разбиение на выборки);
- **`pd.Grouper` и атрибуты дат** — для временных рядов (по годам, месяцам, неделям).


In [ ]:
# 1) Группировка по ДВУМ столбцам: ключ — пара (континент, год). Среднее lifeExp по каждой паре;
#    unstack переносит годы в столбцы; iloc[:, [0, -1]] оставляет первый и последний год.
print(panel.groupby(['continent', 'year'])['lifeExp'].mean().unstack().round(1).iloc[:, [0, -1]])
# 2) Группировка по УРОВНЮ ИНДЕКСА: сначала ставим (континент, год) в индекс, затем level='year'
#    группирует строки по годам (суммарное население мира по пятилеткам).
panel.set_index(['continent', 'year']).groupby(level='year')[['pop']].sum().head(3)

In [ ]:
# 3) Группировка ФУНКЦИЕЙ: она применяется к каждой МЕТКЕ ИНДЕКСА, результат становится ключом группы.
s = pd.Series([1, 2, 3, 4, 5], index=['a', 'bb', 'cc', 'ddd', 'eeeee'])
print(s.groupby(len).sum().to_dict())               # len(метка) = длина названия: группы 1, 2, 3, 5 букв
# Пример на реальных данных: страна в индексе, ключ — первая буква названия страны.
print(panel.set_index('country')['pop'].groupby(lambda name: name[0]).size().head(4).to_dict())

# 4) Группировка булевым Series: две группы — «ВВП выше медианы» и «ниже медианы».
#    rename задаёт понятное имя ключу, оно появится в индексе результата.
rich = (panel['gdpPercap'] > panel['gdpPercap'].median()).rename('gdp_above_median')
panel.groupby(rich)['lifeExp'].agg(['count', 'mean']).round(1)    # число строк и средняя продолжительность жизни

In [ ]:
# 5) Группировка по ДАТАМ. Ряд CO2 имеет индекс-даты (еженедельные наблюдения).
c = co2['co2']
# pd.Grouper(freq='YE') делит ряд на календарные годы (метка — конец года); берём средний уровень CO2.
print(c.groupby(pd.Grouper(freq='YE')).mean().head(3).round(1).to_dict())
# Группировка по атрибуту дат: c.index.month_name() — название месяца каждого наблюдения.
# reindex(months) упорядочивает месяцы по календарю (по умолчанию они шли бы по алфавиту).
months = ['January', 'February', 'March', 'April', 'May', 'June', 'July',
          'August', 'September', 'October', 'November', 'December']
c.groupby(c.index.month_name()).mean().reindex(months).round(1)      # сезонный цикл: максимум весной, минимум осенью

### Параметры группировки

| Параметр | По умолчанию | Смысл |
|---|---|---|
| `sort` | `True` | сортировать группы по ключу (`False` — в порядке появления) |
| `as_index` | `True` | ключи в индексе (`False` — обычные столбцы) |
| `dropna` | `True` | строки с `NaN` в ключе исключаются (`False` — образуют группу `NaN`) |
| `observed` | `True` | для категориальных ключей — только реально встретившиеся категории |

Порядок групп задаётся **упорядоченной категорией**: результат идёт по смыслу, а не по алфавиту.


Смысл параметров на примерах этого ноутбука: `dropna=False` нужен, чтобы увидеть, **сколько** объектов вообще не попало ни в одну группу (страны без группы); `as_index=False` удобен, когда результат дальше соединяют с другой таблицей по столбцу-ключу; `sort=False` сохраняет порядок первого появления (полезно для порядка «по смыслу»); `observed` касается только категориальных ключей: при `observed=True` (значение по умолчанию в 3.0) в результат попадают лишь категории, которые встретились в данных, при `False` — все объявленные, в том числе пустые.


In [ ]:
# dropna. По умолчанию строки с NaN в ключе (страны вне групп) из результата ИСКЛЮЧАЮТСЯ.
print(panel.groupby('primary_bloc')['pop'].size().sum(), 'строк в группах из', len(panel))   # 324 из 1704
# dropna=False выделяет такие строки в отдельную группу NaN (1380 строк).
print(panel.groupby('primary_bloc', dropna=False)['pop'].size().to_dict())

# as_index=False: ключ остаётся обычным столбцом; sort=False: группы в порядке первого появления в данных.
print(panel.groupby('continent', as_index=False, sort=False)['pop'].size().head(3))

# Упорядоченная категория задаёт собственный порядок групп (не алфавитный):
# континенты по убыванию продолжительности жизни.
order = ['Oceania', 'Europe', 'Americas', 'Asia', 'Africa']
p_ord = panel.assign(continent=pd.Categorical(panel['continent'], categories=order, ordered=True))
p_ord.groupby('continent')['lifeExp'].agg(['size', 'mean']).round(1)

### Группировка столбцов (замена `groupby(axis=1)`)

В pandas 3.0 параметра `axis` у `groupby` нет, поэтому сгруппировать **столбцы** (например, по типу данных) методом `groupby` нельзя. Работайте с названиями столбцов и их типами: `df.dtypes`, `df.select_dtypes(...)`, `df.columns.groupby(...)`.

In [ ]:
# Группировка СТОЛБЦОВ по типу данных (замена groupby(axis=1)).
# panel.dtypes — Series «имя столбца -> тип»; приводим типы к строкам для использования как ключ.
dtype_names = panel.dtypes.astype(str)
# Index.groupby(ключи) возвращает словарь «ключ -> названия столбцов этой группы».
print(panel.columns.groupby(dtype_names.tolist()))
# Для выбора столбцов одного типа удобнее select_dtypes.
panel.select_dtypes('number').columns.tolist()[:5]

## 8. Агрегирование групп

**Агрегирование** сворачивает каждую группу в одно значение. Встроенные агрегаты: `size`, `count`, `nunique`; `sum`, `mean`, `median`, `prod`; `min`, `max`, `idxmin`, `idxmax`, `first`, `last`; `std`, `var`, `sem`; `quantile`, `skew`; `any`, `all`.

Метод `agg` (псевдоним `aggregate`) принимает:
- строку с именем метода — самый быстрый вариант;
- список функций `['mean', 'std']`;
- список пар `[('имя', функция), ...]` — задаёт названия столбцов;
- словарь «столбец → функция или список»;
- **именованную агрегацию** `agg(имя=('столбец', функция))` — самый читаемый способ.


**Как устроен результат.** Агрегирование возвращает таблицу, где **одна строка соответствует одной группе** (или паре ключей — тогда индекс составной). Если для столбца задано несколько функций, столбцы результата получают два уровня заголовков (столбец -> функция); именованная агрегация этого избегает, создавая «плоские» столбцы с вашими названиями.

Список встроенных агрегатов можно вызывать двумя способами: методом (`g['x'].mean()`) или строкой в `agg` (`g['x'].agg('mean')`). Строка предпочтительнее для набора функций (`agg(['mean', 'std'])`) и в pandas 3.0 рекомендуется вместо передачи встроенных функций Python (`sum`) или NumPy (`np.mean`).


In [ ]:
# Объект группировки: ключ — пара (континент, год); работаем со столбцом lifeExp.
by_cy = panel.groupby(['continent', 'year'])['lifeExp']

# Собственная функция агрегирования: получает Series значений группы, возвращает ОДНО число.
def peak_to_peak(arr):
    return arr.max() - arr.min()

print(by_cy.agg == by_cy.aggregate)           # agg и aggregate — один и тот же метод (псевдонимы)
# Список из пар (имя, функция) задаёт названия столбцов результата; строка 'std' берёт готовый метод.
# .loc[[(...), ...]] выбирает четыре нужных строки MultiIndex: Африка и Европа в 1952 и 2007 годах.
by_cy.agg([('range', peak_to_peak), ('mean', 'mean'), 'std']).round(1).loc[[('Africa', 1952), ('Africa', 2007), ('Europe', 1952), ('Europe', 2007)]]

In [ ]:
# Словарь «столбец -> функция(и)»: разные агрегаты для разных столбцов.
# Несколько столбцов выбираются ДВОЙНЫМИ квадратными скобками [[...]] (кортеж в pandas 3.0 не допускается).
# Для lifeExp задан список пар (имя, функция) -> столбцы min и max; для pop — одна функция 'sum'.
panel.groupby('continent')[['lifeExp', 'pop']].agg(
    {'lifeExp': [('min', 'min'), ('max', 'max')], 'pop': 'sum'}
).round(1)

In [ ]:
# Именованная агрегация: agg(имя_результата=('столбец', функция)). Самый читаемый способ.
# Функцией может быть строка, def-функция или lambda; названия столбцов результата задаём сами.
(panel.query('year == 2007').groupby('continent').agg(
    n=('country', 'size'),                              # число стран (строк) в группе
    avg_life=('lifeExp', 'mean'),                       # средняя продолжительность жизни
    life_range=('lifeExp', peak_to_peak),               # размах: собственная функция, определённая выше
    total_pop=('pop', 'sum'),                           # суммарное население
    gdp_q90=('gdpPercap', lambda s: s.quantile(0.9)),   # 90-й процентиль ВВП на душу: lambda для разовой задачи
).round(1))

> **Ловушка.** Если в списке `agg([...])` две `lambda`-функции, у обеих будет имя `<lambda>`, и столбцы нельзя различить. Используйте `def` с понятным именем или именованную агрегацию.

### Пропуски в группах

`size()` считает **строки** (с пропусками), `count()` — **непустые значения** столбца. Остальные агрегаты (`mean`, `sum`, `std`, ...) по умолчанию пропуски пропускают; в pandas 3.0 это управляется параметром `skipna`. Пропуски есть в ряде CO₂ (59 недель).


**Почему пропуски важны.** Разные агрегаты по-разному относятся к пропускам: `size` их учитывает (считает строки), `count`, `mean`, `sum`, `std` и др. — игнорируют. Поэтому «среднее по группе» может быть построено на разном числе наблюдений. Практика: рядом со средним всегда полезно выводить `count`, чтобы видеть, на скольких наблюдениях оно основано.


In [ ]:
# Пропуски в группах на ряде CO2: группы — календарные годы.
cy = co2['co2']                                    # ряд CO2 (пригодится и в следующих разделах)
by_year = cy.groupby(cy.index.year)                # ключ — год из индекса-даты
# size() считает ВСЕ наблюдения года, count() — только непустые; их разность = число пропусков.
gaps = pd.DataFrame({'size': by_year.size(), 'count': by_year.count(), 'mean': by_year.mean().round(2)})
gaps.assign(missing=gaps['size'] - gaps['count']).query('missing > 0').head(5)    # годы с пропусками

In [ ]:
# pandas >= 3.0: параметр skipna у методов groupby (mean, sum, median, std, ...).
# skipna=True (по умолчанию) — пропуски игнорируются; skipna=False — любой пропуск в группе даёт NaN.
print(by_year.mean(skipna=True).round(2).head(3).to_dict())
print(by_year.mean(skipna=False).round(2).head(3).to_dict())     # 1958 год содержит пропуски -> NaN

## 9. Статистики по группам: центр, разброс, форма, связь

Описательные статистики вычисляются тем же способом, что и агрегаты: достаточно вызвать их у объекта группировки. Все они по умолчанию пропускают `NaN`.

| Свойство | Функции | Замечания |
|---|---|---|
| Центр | `mean`, `median`, `mode`, `quantile(q)`, `idxmax` | среднее чувствительно к выбросам, медиана устойчива; мод может быть несколько |
| Разброс | `var`, `std`, `sem`, IQR = $Q_3 - Q_1$, коэффициент вариации $s/\bar x$ | $\mathrm{sem} = s/\sqrt n$ |
| Форма | `skew`, `kurt` | $\mathrm{skew} > 0$ — «хвост» вправо; для нормального распределения `kurt` около 0 |
| Сводка | `describe()` | счётчик, среднее, std, min, квартили, max |
| Частоты | `value_counts(normalize=True)` | доли категорий |

**Важно.** В pandas `var()` и `std()` используют `ddof=1` (делитель $n-1$), в NumPy `np.var()`/`np.std()` — `ddof=0` (делитель $n$).

Дальше работаем с данными за 2007 год (`p07`): по одной строке на страну.


**Как читать результаты.**
- Если **среднее заметно больше медианы**, распределение скошено вправо: есть небольшое число очень больших значений (богатые страны, экстремальные цены). В таких случаях описывают данные медианой и IQR.
- **Стандартное отклонение** выражено в единицах признака; **коэффициент вариации** (std/среднее) безразмерен и позволяет сравнивать разброс признаков и групп с разными масштабами.
- **`skew` > 0** — длинный правый «хвост»; **`kurt` > 0** — хвосты тяжелее нормальных (больше экстремальных значений).
- На малых группах (две страны Океании) статистики **ненадёжны**: смотрите на `count`.


In [ ]:
# Для раздела 9 берём срез 2007 года: по одной строке на страну (142 наблюдения).
p07 = panel.query('year == 2007')
g = p07.groupby('continent')['gdpPercap']      # объект группировки: ВВП на душу населения по континентам
# Показатели центра и положения, собранные в одну таблицу (каждый — Series с индексом «континент»).
pd.DataFrame({
    'mean': g.mean(), 'median': g.median(),                 # среднее и медиана: заметная разница = асимметрия/выбросы
    'mode': g.agg(lambda s: s.mode().iloc[0]),              # мода: Series (их может быть несколько) -> берём первую
    'q25': g.quantile(0.25), 'q75': g.quantile(0.75),       # нижний и верхний квартили
}).round(0)

In [ ]:
# idxmax() возвращает МЕТКУ строки с максимумом внутри каждой группы (не само значение).
# По этим меткам .loc выбирает целые строки: получаем страну-рекордсмена для каждого континента.
p07.loc[p07.groupby('continent')['lifeExp'].idxmax(), ['continent', 'country', 'lifeExp', 'gdpPercap']]

In [ ]:
# Межквартильный размах: разность 75-го и 25-го процентилей (устойчивая мера разброса).
def iqr(s):
    return s.quantile(0.75) - s.quantile(0.25)

# Разброс и форма распределения ВВП на душу населения по континентам — одной именованной агрегацией:
#   std, var — стандартное отклонение и дисперсия (ddof=1); sem — стандартная ошибка среднего;
#   iqr — межквартильный размах; cv — коэффициент вариации (отношение std к среднему);
#   skew — асимметрия (>0: длинный «хвост» больших значений); kurt — эксцесс (тяжесть хвостов).
p07.groupby('continent').agg(
    std=('gdpPercap', 'std'), var=('gdpPercap', 'var'), sem=('gdpPercap', 'sem'),
    iqr=('gdpPercap', iqr), cv=('gdpPercap', lambda s: s.std() / s.mean()),
    skew=('gdpPercap', 'skew'), kurt=('gdpPercap', lambda s: s.kurt()),
).round(2)

In [ ]:
# Разница между pandas и NumPy: по умолчанию pandas использует делитель n-1 (ddof=1), NumPy — n (ddof=0).
x = p07['lifeExp']
# Выборочная дисперсия «вручную»: сумма квадратов отклонений от среднего, делённая на (n-1).
print('pandas var       :', round(x.var(), 6), '| по формуле n-1:', round(((x - x.mean()) ** 2).sum() / (len(x) - 1), 6))
# NumPy: np.var по умолчанию ddof=0 (меньшее значение); ddof=1 даёт то же, что pandas.
print('np.var (ddof=0)  :', round(np.var(x), 6), '| np.var(ddof=1):', round(np.var(x, ddof=1), 6))
# Параметр ddof доступен и у методов groupby: сравните стандартные отклонения по континентам.
print(p07.groupby('continent')['lifeExp'].std().round(3).to_dict())              # ddof=1 (по умолчанию)
print(p07.groupby('continent')['lifeExp'].std(ddof=0).round(3).to_dict())        # ddof=0

In [ ]:
# describe() по группам: count, mean, std, min, квартили и max для каждого континента одной командой.
print(p07.groupby('continent')['lifeExp'].describe().round(1))
# Доли классов внутри каждого континента: value_counts(normalize=True) даёт доли (в сумме 1 по континенту);
# unstack разворачивает классы в столбцы; fill_value=0 заменяет отсутствующие комбинации нулём.
p07.groupby('continent')['life_class'].value_counts(normalize=True).unstack(fill_value=0).round(2)

In [ ]:
# Выбросы ВНУТРИ группы по правилу 1.5 x IQR: границы для каждого континента свои.
gdp = p07.groupby('continent')['gdpPercap']
# transform возвращает Series ТОЙ ЖЕ ДЛИНЫ, что и p07: в каждой строке стоит квартиль её континента.
q1, q3 = gdp.transform(lambda s: s.quantile(0.25)), gdp.transform(lambda s: s.quantile(0.75))
low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)         # нижняя и верхняя границы «нормы» для каждой строки
is_outlier = (p07['gdpPercap'] < low) | (p07['gdpPercap'] > high)   # булева маска: значение вне границ
print('выбросов по континентам:', p07[is_outlier].groupby('continent').size().to_dict())
# Самые «богатые» выбросы: сортировка по убыванию ВВП, первые пять.
p07.loc[is_outlier, ['country', 'continent', 'gdpPercap']].sort_values('gdpPercap', ascending=False).head(5)

### Ковариация и корреляция

**Ковариация** показывает, как два признака изменяются вместе, но зависит от единиц измерения. **Коэффициент корреляции Пирсона**

$$
r_{xy} = \frac{\mathrm{cov}(x, y)}{s_x \, s_y}, \qquad -1 \le r_{xy} \le 1 .
$$

- $r$ близок к 1 — сильная положительная линейная связь, к −1 — отрицательная, к 0 — линейной связи нет;
- `method='spearman'` считает корреляцию рангов (подходит для монотонных нелинейных связей);
- корреляция **не означает причинности**.

Методы: `Series.cov/corr(other)`, `DataFrame.cov()`, `DataFrame.corr(numeric_only=True)`; для групп — `groupby(...)[cols].corr()`.


**Как интерпретировать.** Ориентировочная шкала для $|r|$: до 0.3 — слабая связь, 0.3–0.7 — умеренная, выше 0.7 — сильная (границы условны и зависят от области). Три предупреждения: (1) корреляция измеряет **только линейную** связь; нелинейную зависимость она может недооценить; (2) на значение сильно влияют **выбросы** и **малый размер выборки**; (3) корреляция внутри групп может отличаться от общей, поэтому группировать данные перед анализом связей полезно.


In [ ]:
cols = ['lifeExp', 'gdpPercap']
# Ковариация и корреляция Пирсона двух столбцов. Ковариация зависит от единиц измерения, корреляция — нет.
print('cov :', round(panel[cols[0]].cov(panel[cols[1]]), 2), '| corr:', round(panel[cols[0]].corr(panel[cols[1]]), 4))
# Проверка формулы r = cov / (std_x * std_y): результат должен совпасть с corr.
print('cov / (std * std):', round(panel[cols[0]].cov(panel[cols[1]]) / (panel[cols[0]].std() * panel[cols[1]].std()), 4))
# Матрица корреляций для трёх столбцов (диагональ равна 1, матрица симметрична).
panel[['lifeExp', 'pop', 'gdpPercap']].corr().round(3)

In [ ]:
# Сравниваем три меры связи ВВП и продолжительности жизни:
#   Пирсон — линейная связь; Спирмен — связь рангов (любая монотонная зависимость);
#   Пирсон с логарифмом ВВП — линейная связь после «сжатия» шкалы дохода.
print('Пирсон              :', round(panel['lifeExp'].corr(panel['gdpPercap']), 3))
print('Спирмен             :', round(panel['lifeExp'].corr(panel['gdpPercap'], method='spearman'), 3))
print('Пирсон с log(ВВП)   :', round(panel['lifeExp'].corr(np.log(panel['gdpPercap'])), 3))

Корреляции Спирмена (по рангам) и Пирсона с **логарифмом** ВВП заметно выше обычной корреляции Пирсона: связь монотонная, но не линейная — прирост продолжительности жизни от дополнительного дохода убывает по мере роста дохода. Поэтому выбор метода и преобразования признака влияет на вывод.

In [ ]:
# Корреляция внутри групп двумя способами (результаты совпадают).
# Способ 1: groupby(...)[cols].corr() даёт матрицу 2x2 для каждого континента; xs выбирает строки lifeExp,
#           а ['gdpPercap'] — нужный элемент матрицы.
print(panel.groupby('continent')[cols].corr().round(3).xs('lifeExp', level=1)['gdpPercap'].to_dict())
# Способ 2: apply с lambda — функция получает подтаблицу континента и считает корреляцию столбцов.
panel.groupby('continent')[cols].apply(lambda d: d['lifeExp'].corr(d['gdpPercap'])).round(3).to_dict()

Внутри континентов связь выражена по-разному. Коэффициент для Океании рассчитан по двум странам (24 наблюдения: по 12 пятилетних замеров на страну), поэтому он малонадёжен — число наблюдений в группе нужно учитывать при интерпретации.

## 10. Преобразование групп: `transform`, накопительные функции, `filter`, `apply`

| Метод | Что возвращает | Задача |
|---|---|---|
| `agg` | по строке на группу | таблица статистик |
| `transform` | результат той же формы, что исходные данные | нормирование и заполнение пропусков внутри группы, доля от суммы группы |
| `filter` | исходные строки тех групп, для которых функция вернула `True` | оставить крупные или «полные» группы |
| `apply` | произвольный результат для группы | топ-N внутри группы |

Если задачу можно решить через `agg`, `transform` или `filter`, предпочитайте их: они проще и быстрее, чем `apply`.

Накопительные и разностные функции сохраняют длину и индекс, поэтому хорошо работают внутри групп: `cumsum`, `cumprod`, `cummax`, `rank(method=...)`, `diff(k)`, `shift(k)`, `pct_change(k)` $= x_t/x_{t-k} - 1$.

### 10.1 `transform`


**Как выбрать метод.** Задайте себе вопрос о желаемом результате:
- нужна *одна строка на группу* -> `agg`;
- нужен *новый столбец той же длины, что таблица* (нормированное значение, доля от суммы группы, ранг внутри группы) -> `transform`;
- нужно *убрать целые группы по условию* -> `filter`;
- нужен *произвольный, более сложный результат на группу* -> `apply`.

`transform` особенно полезен потому, что его результат **выравнивается по исходному индексу**, поэтому им можно напрямую заполнять новые столбцы и использовать в булевых масках.


In [ ]:
# transform: нормирование (z-оценка) ВНУТРИ каждого года. Для каждой строки: (значение - среднее года) / std года.
# Результат — Series той же длины, что panel (по одному значению на строку).
z = panel.groupby('year')['lifeExp'].transform(lambda s: (s - s.mean()) / s.std())
# Проверка: после нормирования в каждом году среднее равно 0, а стандартное отклонение — 1.
print(z.groupby(panel['year']).agg(['mean', 'std']).round(6).head(3))

In [ ]:
# Доля страны в населении своего континента в данном году: делим население на сумму по группе.
# transform('sum') «растягивает» сумму группы на все её строки, поэтому деление выполняется построчно.
grp = panel.groupby(['continent', 'year'])['pop']
share = (panel['pop'] / grp.transform('sum')).round(4)
# rank внутри года: 1 = наибольшая продолжительность жизни (ascending=False), метод по умолчанию 'average'.
rank = panel.groupby('year')['lifeExp'].rank(ascending=False)
# Смотрим Океанию в 2007 году: доли двух стран в сумме дают 1.
panel.assign(pop_share=share, rank_in_year=rank).query('year == 2007 and continent == "Oceania"')[
    ['country', 'pop', 'pop_share', 'lifeExp', 'rank_in_year']]

In [ ]:
# Заполнение пропусков средним ЗНАЧЕНИЕМ СВОЕЙ ГРУППЫ (группы — годы ряда CO2).
# transform('mean') возвращает ряд той же длины, где каждое наблюдение заменено средним своего года;
# fillna подставляет это среднее ТОЛЬКО на место пропусков.
filled = cy.fillna(cy.groupby(cy.index.year).transform('mean'))
print('Пропусков было:', cy.isna().sum(), '| стало:', filled.isna().sum())    # ожидаем 59 -> 0

### 10.2 Накопительные и разностные показатели внутри групп


Функции `diff`, `pct_change`, `cumsum`, `cummax`, `shift`, `rank` по умолчанию работают **по порядку строк**, поэтому перед их применением данные должны быть упорядочены (в Gapminder — по годам внутри страны, в `stocks_long` — по дате). Применённые к объекту `groupby`, они начинают отсчёт **заново в каждой группе**: у первой строки группы разность и процентное изменение равны `NaN`, а накопленные значения стартуют с нуля.


In [ ]:
# Накопительные и разностные показатели ВНУТРИ групп (группа = страна). Данные упорядочены по годам,
# поэтому «предыдущее значение» — это предыдущая пятилетка той же страны.
by_c = panel.groupby('country')['lifeExp']
res = panel.assign(
    diff=by_c.diff().round(2),                           # разность с предыдущим замером (у первого года — NaN)
    pct=(by_c.pct_change() * 100).round(1),              # относительный рост в процентах
    cummax=by_c.cummax(),                                # накопленный максимум с начала ряда
    since_start=(panel['lifeExp'] - by_c.transform('first')).round(1))   # прирост относительно 1952 года
res.query('country == "Japan"')[['year', 'lifeExp', 'diff', 'pct', 'cummax', 'since_start']].head(4)

In [ ]:
# Недельные доходности акций по каждой компании. pct_change нужно считать ВНУТРИ групп-тикеров,
# иначе первая неделя одной компании «сравнивалась» бы с последней неделей предыдущей.
sl = stocks_long.sort_values(['ticker', 'date'])           # внутри компании — по времени
sl = sl.assign(ret=sl.groupby('ticker')['price'].pct_change())   # ret = P_t / P_(t-1) - 1
# Средняя доходность и волатильность (стандартное отклонение доходностей) по компаниям.
sl.groupby('ticker')['ret'].agg(mean='mean', volatility='std').round(4).sort_values('volatility')

In [ ]:
# Как rank обрабатывает совпадающие значения (165 встречается дважды):
#   average — среднее мест (2.5 и 2.5); min — наименьшее место (2 и 2); dense — «плотный» ранг без пропусков;
#   first — по порядку появления (2 и 3).
h = pd.Series([160, 165, 165, 170, 175], index=list('abcde'))
pd.DataFrame({'value': h, 'average': h.rank(), 'min': h.rank(method='min'),
              'dense': h.rank(method='dense'), 'first': h.rank(method='first')})

### 10.3 `filter`: группа остаётся или исчезает целиком


Отличие от обычной фильтрации строк: булева маска `df[df['x'] > 5]` отбирает **строки** по их собственному значению, а `filter` отбирает **группы целиком** по свойству группы (размер, число непустых значений, среднее). Функция, передаваемая в `filter`, получает подтаблицу группы и обязана вернуть **одно** значение `True` или `False`.


In [ ]:
# filter: функция получает подтаблицу группы и возвращает True/False; группа остаётся ЦЕЛИКОМ или исчезает.
d = pd.DataFrame({'Label': list('AABCCC'), 'Values': [1, 2, 3, 4, np.nan, 8]})
# Оставляем группы, где не менее двух непустых значений (count не считает NaN): остаются A и C (группа B исчезает).
print(d.groupby('Label').filter(lambda g: g['Values'].count() > 1))
# Оставляем группы без пропусков: остаются A и B (в группе C есть NaN).
print(d.groupby('Label').filter(lambda g: g['Values'].isna().sum() == 0))
# На реальных данных: континенты, где не менее 30 различных стран (nunique — число уникальных значений).
big_cont = panel.groupby('continent').filter(lambda g: g['country'].nunique() >= 30)
print('континенты не менее чем с 30 странами:', sorted(big_cont['continent'].unique()))

### 10.4 `groupby.apply`

Функция получает группу целиком (`DataFrame`) и результаты склеиваются. **В pandas 3.0** столбцы-ключи в неё не передаются, параметр `include_groups=True` запрещён; значение ключа — `g.name`. Явно выбирайте нужные столбцы: `groupby(key)[['a', 'b']].apply(...)`.


**Форма результата.** Если функция возвращает `DataFrame` с метками строк (как `nlargest`), результаты групп склеиваются, а ключ группы становится верхним уровнем индекса. Если функция возвращает `Series` (как в следующем примере), получается таблица «одна строка на группу». Если возвращается число — `Series` с индексом групп. Перед использованием `apply` проверьте, нельзя ли решить задачу через `agg`, `transform` или `filter`: они быстрее и проще.


In [ ]:
# Функция для groupby.apply: получает ПОДТАБЛИЦУ группы, возвращает её top-n строк по выбранному столбцу.
# (Столбец-ключ 'continent' в pandas 3.0 в подтаблицу не передаётся — но он и не нужен.)
def top(d, n=3, column='lifeExp'):
    return d.sort_values(by=column).tail(n)          # сортировка по возрастанию, tail(n) берёт n наибольших

# Явно выбираем нужные столбцы двойными скобками; n=2 передаётся в top как именованный аргумент.
# Результат: две страны с наибольшей продолжительностью жизни в каждом континенте
# (ключ континента + исходная метка строки образуют MultiIndex результата).
p07.groupby('continent')[['country', 'lifeExp', 'gdpPercap']].apply(top, n=2).round(1)

In [ ]:
# Ключ группы доступен как g.name. Функция возвращает Series -> результаты групп складываются в таблицу.
p07.groupby('continent')[['pop']].apply(
    lambda g: pd.Series({'continent': g.name,                       # название континента (ключ группы)
                         'n': len(g),                               # число стран в группе
                         'max_pop_mln': round(g['pop'].max() / 1e6)})   # максимальное население, млн
)

## 11. Оконные функции и временные ряды

**Окно** — подмножество соседних значений, для которого считается статистика:
- `rolling(window)` — скользящее окно: `mean`, `sum`, `std`, `min`, `max`, `median`, `apply(func, raw=True)`;
- `expanding()` — окно растёт от начала ряда;
- `ewm(span=...)` — экспоненциально взвешенное окно: недавние значения важнее;
- `min_periods` — минимальное число наблюдений в окне, иначе `NaN` (по умолчанию окно требует **всех** `window` значений, поэтому пропуски в данных дают `NaN` в результате);
- `resample(freq)` — агрегирование временного ряда по периодам (для `DatetimeIndex`).

Первые `window - 1` значений скользящей статистики по умолчанию равны `NaN`. Внутри групп оконные функции вызывают через `transform`: так окно не «перетекает» из одной группы в другую.

Ряд CO₂ содержит пропуски, поэтому он хорошо показывает роль `min_periods`.


**Как считается окно.** Окно `rolling(3)` «заканчивается» в текущем наблюдении и включает его и два предыдущих; на каждом шаге оно сдвигается на одну позицию. Для еженедельного ряда `rolling(52)` — это среднее за последние 52 недели (примерно год), оно сглаживает сезонные колебания и показывает тренд. Параметр `center=True` центрирует окно вокруг текущего наблюдения.

В `ewm(span=N)` вес наблюдения убывает по экспоненте с возрастом; параметр сглаживания $\alpha = 2/(N+1)$.

**`rolling` и `resample` решают разные задачи.** `rolling` оставляет ряд той же длины и частоты (значение для каждого наблюдения), а `resample` **меняет частоту** (сводит недели в годы, месяцы и т. д.) и возвращает по одной строке на период.


In [ ]:
# rolling(3) создаёт объект «скользящее окно из 3 наблюдений»; вычисление запускает метод (mean).
r = cy.rolling(3)
# Проверка определения: ищем первое окно без пропусков и сравниваем с обычным средним трёх значений.
i = r.mean().index.get_loc(r.mean().first_valid_index())
print(type(r).__name__, '| первое значение rolling(3):', round(r.mean().iloc[i], 4), '=', round(cy.iloc[i - 2:i + 1].mean(), 4))

# Сравнение вариантов окон на ряде с пропусками (почти 44 года, еженедельно):
#   roll_52        — окно 52 недели, требуется ВСЕ 52 значения (любой пропуск в окне даёт NaN);
#   roll_52_min26  — достаточно 26 непустых значений в окне (min_periods=26);
#   expanding      — окно растёт от начала ряда (накопленное среднее);
#   ewm_52         — экспоненциально взвешенное среднее: недавние значения весят больше.
trend = pd.DataFrame({
    'co2': cy,
    'roll_52': cy.rolling(52).mean(),
    'roll_52_min26': cy.rolling(52, min_periods=26).mean(),
    'expanding': cy.expanding().mean(),
    'ewm_52': cy.ewm(span=52).mean(),
}).round(2)
# count() показывает, сколько значений удалось вычислить: min_periods заметно сокращает число NaN.
print('значений roll_52 / roll_52_min26:', trend['roll_52'].count(), '/', trend['roll_52_min26'].count())
trend.iloc[[0, 30, 100, 1000, -1]]        # несколько строк по ходу ряда (начало, середина, конец)

In [ ]:
# rolling(...).apply: собственная функция в окне. raw=True передаёт в функцию массив NumPy (быстрее Series).
# Из-за пропусков внутри окна используем np.nanmax/np.nanmin (игнорируют NaN). Результат — размах в окне 52 недели.
# dropna() убирает первые значения, где окно ещё не накопило min_periods наблюдений.
cy.rolling(52, min_periods=26).apply(lambda w: np.nanmax(w) - np.nanmin(w), raw=True).dropna().head(3).round(2).to_dict()

In [ ]:
# Оконные функции ВНУТРИ групп-лет: окно не переходит границу года, потому что вычисляется отдельно для каждой группы.
by_year_s = cy.groupby(cy.index.year)
within = pd.DataFrame({
    'co2': cy,
    # скользящее среднее по 4 неделям; min_periods=1 — чтобы в начале каждого года не было NaN
    'roll4_in_year': by_year_s.transform(lambda s: s.rolling(4, min_periods=1).mean()),
    # накопленное среднее с начала года (в январе следующего года счёт начинается заново)
    'expanding_in_year': by_year_s.transform(lambda s: s.expanding().mean()),
}).round(2)
within.loc['1960'].head(4)               # первые недели 1960 года (выбор по части даты)

In [ ]:
# resample: изменение частоты ряда (еженедельные данные -> годовые). 'YE' — конец года.
# agg считает несколько показателей за каждый год сразу.
annual = cy.resample('YE').agg(['mean', 'max', 'min'])
print(annual.head(3).round(1))
# resample('YE') и groupby(pd.Grouper(freq='YE')) дают один и тот же результат. Ожидаем True.
print('Совпадает с Grouper:', annual['mean'].equals(cy.groupby(pd.Grouper(freq='YE')).mean()))
# Годовой рост среднего уровня CO2 в процентах (pct_change по годовому ряду) и его описательная статистика.
annual['mean'].pct_change().mul(100).round(2).describe().round(2)

## 12. Интервалы и сводные таблицы

### 12.1 Дискретизация: `pd.cut` и `pd.qcut`

- `pd.cut(x, bins, labels)` — интервалы заданной ширины или с заданными границами (`bins=5` делит диапазон на 5 **равных по ширине** частей);
- `pd.qcut(x, q)` — интервалы с **равным числом наблюдений** (по квантилям).

Интервалы по умолчанию закрыты справа: $(a, b]$; `include_lowest=True` включает левую границу первого интервала; `retbins=True` возвращает границы. Результат — упорядоченный категориальный столбец, по которому можно группировать.


**Границы интервалов.** Для `cut` с границами `[0, 50, 65, 75, 100]` получаются интервалы $(0, 50]$, $(50, 65]$, $(65, 75]$, $(75, 100]$: значение 50 попадёт в первый интервал, а 65 — во второй. Значения вне границ получают `NaN`. У `qcut` границы вычисляются автоматически по квантилям, поэтому они зависят от данных (их можно получить через `retbins=True`). Результат обеих функций — **упорядоченный категориальный** столбец, поэтому по нему можно группировать, а порядок групп будет логичным.


In [ ]:
# --- cut и qcut на искусственных данных: 10 000 значений из нормального распределения ---
rng = np.random.default_rng(123456)             # генератор случайных чисел с фиксированным начальным значением
dist = rng.normal(size=10000)
# cut(dist, 5): 5 интервалов РАВНОЙ ШИРИНЫ — по краям очень мало наблюдений, в центре много.
print('cut :', pd.cut(dist, 5).value_counts().sort_index().tolist(), '(ширина одинакова, число разное)')
# qcut(dist, 5): 5 интервалов с РАВНЫМ ЧИСЛОМ наблюдений (по 2000), ширина разная.
print('qcut:', pd.qcut(dist, 5).value_counts().sort_index().tolist(), '(число одинаковое)')

# --- на данных 2007 года ---
# Квартили ВВП на душу: labels задаёт названия, retbins=True дополнительно возвращает границы интервалов.
gdp_q, edges = pd.qcut(p07['gdpPercap'], 4, labels=['q1', 'q2', 'q3', 'q4'], retbins=True)
# Классы продолжительности жизни по заданным границам (интервалы вида (a, b]).
life_bin = pd.cut(p07['lifeExp'], bins=[0, 50, 65, 75, 100], labels=['<50', '50-65', '65-75', '75+'])
print('границы квартилей ВВП:', edges.round(0))
# Категориальный столбец можно использовать как ключ группировки: средняя продолжительность жизни по квартилям ВВП.
p07.groupby(gdp_q)['lifeExp'].agg(['count', 'mean', 'std']).round(1)

### 12.2 Сводные таблицы: `pivot_table`

`pivot_table(values, index, columns, aggfunc)` строит таблицу: строки и столбцы — значения признаков, в ячейках — агрегат. Это `groupby` по нескольким ключам с последующим `unstack`, но короче. Параметры: `aggfunc` (строка, список или словарь; по умолчанию `'mean'`), `margins=True` (итоги), `fill_value`. **Всегда указывайте `values`**, иначе pandas попытается агрегировать и нечисловые столбцы.


**Связь с `groupby`.** Вызов `df.pivot_table(values='v', index='a', columns='b', aggfunc='mean')` эквивалентен `df.groupby(['a', 'b'])['v'].mean().unstack()`. Дополнительные возможности `pivot_table`: итоги `margins=True` (в строке и столбце `All`), разные функции для разных величин через словарь, замена пропусков `fill_value`. Если нужны итоги по подмножеству или более сложная логика, используют `groupby` напрямую.


In [ ]:
# pivot_table: строки — континенты, столбцы — годы, в ячейках — средняя продолжительность жизни.
# margins=True добавляет итоги: строку 'All' (все континенты) и столбец 'All' (все годы).
# .loc[:, [...]] показывает только три года и итоговый столбец (таблица иначе слишком широкая).
panel.pivot_table(values='lifeExp', index='continent', columns='year', aggfunc='mean',
                  margins=True).round(1).loc[:, [1952, 1982, 2007, 'All']]

In [ ]:
# Разные функции для разных величин: aggfunc в виде словаря «столбец -> функция».
# Среднее — для продолжительности жизни, сумма — для населения; столбцы — квартили ВВП (gdp_q).
pt = p07.assign(gdp_q=gdp_q).pivot_table(values=['lifeExp', 'pop'], index='continent', columns='gdp_q',
                                         aggfunc={'lifeExp': 'mean', 'pop': 'sum'})
print(pt['lifeExp'].round(1))               # NaN: в континенте нет стран этого квартиля ВВП (заменять нулём нельзя)
(pt['pop'] / 1e6).fillna(0).round(0)        # население, млн: для суммы пустая ячейка = 0 жителей, заменяем нулём явно

### 12.3 `crosstab`

`pd.crosstab(a, b)` считает частоты сочетаний; `normalize='index'` / `'columns'` / `True` переводит их в доли; `margins=True` добавляет итоги.


`crosstab` — это таблица **частот** (сколько наблюдений имеют сочетание значений двух признаков); она эквивалентна `pivot_table` с подсчётом числа строк. Параметр `normalize` переводит частоты в доли: `'index'` — доли внутри каждой строки (сумма по строке равна 1), `'columns'` — внутри столбца, `True` — от общего числа.


In [ ]:
# crosstab: таблица частот. normalize='index' переводит числа в ДОЛИ внутри каждой строки (сумма по строке = 1):
# доля стран континента, попавших в каждый квартиль ВВП.
print(pd.crosstab(p07['continent'], gdp_q, normalize='index').round(2))
# Абсолютные частоты континент x класс продолжительности жизни; margins=True добавляет итоги (All).
pd.crosstab(p07['continent'], life_bin, margins=True)

### 12.4 Результат `groupby` + `unstack`

Результат `groupby` по двум ключам — `Series` с `MultiIndex`; `unstack` разворачивает его в таблицу. Если какой-то комбинации нет, появляется `NaN`, который можно заменить через `fill_value`. Для счётчиков нуль — осмысленная замена.


Этот приём — перенос агрегатов `groupby` в таблицу «категория x категория» — используется постоянно; `pivot_table` выполняет те же действия одной командой. Следите за пропусками: нулём (`fill_value=0`) заменяют **счётчики и суммы**, но не средние, медианы и другие характеристики, для которых «нет данных» не равно нулю.


In [ ]:
# Число стран с продолжительностью жизни выше 75 лет по континентам и годам.
# query отбирает подходящие строки, groupby(...).size() считает их число в каждой паре (континент, год).
# Комбинации, где подходящих стран нет, в результате ОТСУТСТВУЮТ.
n_long_lived = panel.query('lifeExp > 75').groupby(['continent', 'year']).size()
cols_y = [1977, 1992, 2007]
# unstack разворачивает годы в столбцы: недостающие комбинации превращаются в NaN.
print(n_long_lived.unstack().loc[:, cols_y])
# fill_value=0 для счётчиков осмысленно: «нет таких стран» = 0.
long_lived = n_long_lived.unstack(fill_value=0)
long_lived.loc[:, cols_y]

### 12.5 Каркас «все комбинации» + факты

Часто нужна таблица со **всеми** комбинациями (группа × год), даже если для части данных нет. Строим каркас через `how='cross'` (раздел 4), присоединяем факты `left`-слиянием и разворачиваем `pivot`.

В этих данных пропусков не возникнет (у каждой группы есть страны в обоих годах), но каркас гарантирует полноту таблицы при любых данных: отсутствующие сочетания появятся как `NaN`, а не пропадут из результата.

In [ ]:
# Задача: получить таблицу «группа x год» СО ВСЕМИ комбинациями, даже если фактов для них нет.
# Шаг 1. Факты: средняя продолжительность жизни в 1952 и 2007 годах для стран каждой группы.
#   melt превращает флаги in_G7, in_G20, ... в пары (bloc, member); query('member == 1') оставляет
#   только страны-члены; str.replace убирает префикс 'in_' из названия группы; затем groupby считает среднее.
facts = (panel.query('year in [1952, 2007]')
         .melt(id_vars=['country', 'year', 'lifeExp'], value_vars=flag_cols, var_name='bloc', value_name='member')
         .query('member == 1')
         .assign(bloc=lambda d: d['bloc'].str.replace('in_', '', regex=False))
         .groupby(['bloc', 'year'], as_index=False)['lifeExp'].mean())

# Шаг 2. Присоединяем факты к каркасу (skeleton из раздела 4): how='left' сохраняет все 8 комбинаций.
table = skeleton.merge(facts, on=['bloc', 'year'], how='left')
# Шаг 3. pivot превращает таблицу в «группа x год»; добавляем столбец прироста (2007 минус 1952).
(table.pivot(index='bloc', columns='year', values='lifeExp')
      .rename_axis(columns=None).round(1)
      .assign(gain=lambda d: (d[2007] - d[1952]).round(1)))

## 13. Задания для самостоятельной работы

Исходные таблицы не изменяйте: используйте `assign` или отдельные переменные. Результаты проверяйте на размерах таблиц.

### Задание 1. Сборка таблицы: `merge`, `join`, `pivot` (разделы 4–5)
Постройте таблицу `panel2` заново из `gapminder`, `continents` и `blocs`. Найдите через `indicator` членов групп, которых нет в Gapminder, исправьте названия там, где причина — разное написание, и объясните причину остальных. Присоедините к стране число групп `n_blocs` через `join`. Проверьте, что число строк осталось равным 1704, а `validate` не выдаёт ошибок. Сравните число строк при `how='left'`, `'inner'` и `'outer'` для слияния `gapminder` с `blocs` и объясните разницу.

In [ ]:
# Подсказки: gapminder.merge(continents, left_on=..., right_on=..., validate='many_to_one');
# blocs.merge(g07[['country']], on='country', how='left', indicator=True);
# country_info.join(n_blocs); сравнение how='left' / 'inner' / 'outer' — через len(...) результата.
# Ваш код здесь

### Задание 2. Форма таблицы: `melt`, `pivot`, `stack` (раздел 2 и 10)
Преобразуйте `stocks` в длинный формат. Верните его в широкий (`pivot`) и вычислите недельные доходности (`pct_change()`) по каждой компании. Постройте матрицу корреляций доходностей и найдите пару компаний с наибольшей корреляцией. Затем для каждой компании найдите неделю максимальной цены и итоговый рост (`groupby`), верните доходности в длинный вид через `stack().dropna()` и сравните число строк с ожидаемым.

In [ ]:
# Подсказки: stocks.melt(id_vars='date', ...); long.pivot(index='date', columns='ticker', values='price');
# wide.pct_change(); .corr(); для пары с наибольшей корреляцией — .unstack() и sort_values (не забудьте убрать диагональ);
# stack().dropna(); groupby('ticker') + idxmax для недели максимума.
# Ваш код здесь

### Задание 3. Функции и агрегирование (разделы 6–8)
Для `panel` за 2007 год создайте столбец `pop_class` через `Series.apply` и функцию: население меньше 5 млн — `'small'`, от 5 до 50 млн — `'medium'`, больше 50 млн — `'large'`. Повторите через `np.select` и проверьте совпадение. Затем именованной агрегацией постройте таблицу по `continent` × `pop_class`: число стран, суммарное население, среднюю и медианную продолжительность жизни и собственную функцию — размах ВВП на душу населения. Сколько стран не состоит ни в одной группе в каждом континенте (`size() - count()` по `primary_bloc`)?

In [ ]:
# Подсказки: panel.query('year == 2007'); .apply(функция) и np.select([...], [...], default=...);
# именованная агрегация .groupby(['continent', 'pop_class']).agg(n=(...), ...);
# число стран без группы в континенте: size() - count() по primary_bloc.
# Ваш код здесь

### Задание 4. Статистика по группам (раздел 9)
Для 2007 года вычислите по континентам: среднее, медиану, std, IQR, коэффициент вариации и асимметрию продолжительности жизни; найдите континент с наибольшим разбросом и объясните, почему среднее и медиана в некоторых группах заметно различаются. Затем вычислите корреляции Пирсона и Спирмена между `lifeExp` и `gdpPercap` внутри каждого континента, а также корреляцию `lifeExp` с `log(gdpPercap)`. Сделайте вывод о форме связи. Найдите выбросы по ВВП внутри континентов по правилу 1.5 × IQR.

In [ ]:
# Подсказки: groupby('continent').agg(...) с функциями mean, median, std, iqr, cv (lambda), skew;
# .corr(method='spearman'); np.log(...); границы выбросов — через transform (см. раздел 9).
# Ваш код здесь

### Задание 5. Преобразования внутри групп (раздел 10)
Для `panel`: а) вычислите продолжительность жизни, взвешенную по населению, для каждой пары «континент — год» и разверните в таблицу (`unstack`), найдите абсолютный прирост 1952–2007; б) для каждой страны вычислите прирост продолжительности жизни от 1952 года (`transform('first')`) и выведите по три страны с наибольшим приростом в каждом континенте (`groupby.apply` и вторым способом — `sort_values` + `groupby.head`); в) оставьте только континенты, где средний по странам прирост продолжительности жизни 1952–2007 больше 20 лет (`filter`); г) вычислите ранг страны по продолжительности жизни внутри года и найдите страны, которые поднялись в рейтинге сильнее всех между 1952 и 2007 годами.

In [ ]:
# Подсказки: взвешенное среднее = (lifeExp * pop).sum() / pop.sum() по группе; unstack; transform('first');
# groupby.apply(...nlargest...) и sort_values + groupby(...).head(3); filter; rank + разность рангов 1952 и 2007.
# Ваш код здесь

### Задание 6. Временной ряд и сводные таблицы (разделы 11–12)
Для ряда `co2`: а) подсчитайте число пропусков по годам (`size() - count()`) и заполните их двумя способами — средним по году (`transform`) и скользящим средним с `min_periods`; сравните результаты; б) вычислите скользящее среднее за 52 недели и годовой размах сезонных колебаний (max − min по годам, через `resample('YE')`); в) постройте `pivot_table` «год × месяц» со средним CO₂ (используйте `index.year` и `index.month`) и найдите месяц с максимумом и минимумом в среднем за все годы; г) разбейте годы на квартили по среднему уровню CO₂ (`qcut`) и найдите в каждом квартиле средний годовой рост (`pct_change`).

In [ ]:
# Подсказки: cy.groupby(cy.index.year); fillna(transform('mean')); cy.rolling(52, min_periods=26).mean();
# cy.resample('YE'); pd.pivot_table(cy.to_frame(), index=cy.index.year, columns=cy.index.month, values='co2');
# pd.qcut(годовые_средние, 4); pct_change().
# Ваш код здесь

## 14. Контрольные вопросы

*Форма и объединение таблиц*
1. Чем длинный формат отличается от широкого? Для каких задач удобен каждый из них?
2. Чем `melt` отличается от `stack`, а `pivot` — от `unstack`?
3. Почему `pivot` может выдать ошибку и чем её устранить? Чем `pivot_table` отличается от `pivot`?
4. Что изменилось в `stack` в pandas 3.0 и как убрать строки с пропусками?
5. Чем `concat` отличается от `merge`? Какие значения принимают `axis` и `join` у `concat`? Для чего нужны `keys`, `ignore_index`, `verify_integrity`?
6. Какие типы соединений поддерживает `merge`? Сколько строк даст ключ, встречающийся 2 раза слева и 3 раза справа?
7. Как найти строки одной таблицы, у которых нет пары в другой? Что делают `indicator` и `validate`?
8. Что происходит с `NaN` в ключах при слиянии? Чем это отличается от SQL?
9. Чем `join` отличается от `merge`? Какое значение `how` по умолчанию у каждого? Как избежать конфликта одноимённых столбцов?

*Новые признаки*

10. В чём разница между `map` со словарём и `map` с функцией? Чем `axis=0` отличается от `axis=1` в `DataFrame.apply`?
11. Почему `apply(axis=1)` по большой таблице медленный и чем его можно заменить?

*Группировка и статистика*

12. Что означает split–apply–combine? Когда выполняется вычисление при `groupby`?
13. Какие способы задания групп вы знаете? Что делают параметры `sort`, `as_index`, `dropna`, `observed`?
14. Чем `size()` отличается от `count()`? Как каждый из них относится к пропускам?
15. Какие формы аргумента принимает `agg`? Чем именованная агрегация удобнее списка функций?
16. Чем `transform` отличается от `agg`? Приведите задачу, где нужен именно `transform`.
17. Что делает `filter`? Что изменилось в `groupby.apply` в pandas 3.0 и как получить значение ключа группы?
18. Чем среднее отличается от медианы? Почему `std()` в pandas и `np.std()` дают разные числа?
19. Что показывает коэффициент корреляции и чего не показывает? Почему корреляция по всей выборке может отличаться от корреляции внутри групп и от корреляции с логарифмом признака?
20. Как работают `rank`, `pct_change` и `rolling` на временном ряде? Какую роль играет `min_periods` при наличии пропусков?
21. Чем `pivot_table` отличается от `groupby` по двум ключам? Чем `cut` отличается от `qcut`?
22. Почему в pandas 3.0 нельзя использовать `df.applymap` и `groupby(axis=1)` и чем их заменить? Почему не нужен параметр `copy`?

## 15. Краткая памятка

```python
# Форма таблицы
df.melt(id_vars=[...], value_vars=[...], var_name='var', value_name='val')
df.pivot(index='i', columns='c', values='v')               # без агрегации, пары уникальны
df.pivot_table(index='i', columns='c', values='v', aggfunc='mean', margins=True)
df.set_index(['a', 'b'])['v'].unstack(level='b', fill_value=0)
wide.stack().dropna()                                        # в pandas 3.0 NaN сохраняются
pd.wide_to_long(df, stubnames=['s'], i='id', j='t', sep='_')

# Объединение
pd.concat([a, b], axis=0, join='outer', ignore_index=True, keys=['a', 'b'], verify_integrity=True)
pd.concat({'a': a, 'b': b}, names=['src', 'row'])
pd.merge(l, r, how='left', on='k', validate='many_to_one', indicator=True, suffixes=('_l', '_r'))
pd.merge(l, r, left_on='x', right_on='y');  pd.merge(l, r, left_index=True, right_index=True)
pd.merge(l, r, how='cross')
l.join(r, how='left', on='k', lsuffix='_l', rsuffix='_r');  l.join([r1, r2])

# Функции
s.map({'a': 1}); s.map(func, na_action='ignore'); s.apply(func, args=(x,), key=y)
df.apply(func, axis=0 / 1); df.map(func); df.assign(c=pd.col('a') + pd.col('b'))   # pd.col: >= 3.0
np.where(cond, a, b); np.select([c1, c2], [v1, v2], default)

# Группировка
g = df.groupby(['k1', 'k2'], sort=True, as_index=True, dropna=True)
g.ngroups; g.size(); g.groups; g.get_group(k); g.nth(0); g.cumcount()
df.groupby(level='lvl'); df.groupby(len); df.groupby(mask); df.groupby(pd.Grouper(freq='YE'))

# Агрегирование и статистика по группам
g['x'].agg(['mean', 'std']); g['x'].agg([('name', func)])
g.agg(out=('x', 'mean'), r=('x', lambda s: s.max() - s.min()))
g['x'].median(); g['x'].quantile(.9); g['x'].std(ddof=0); g['x'].skew(); g['x'].describe()
g['x'].value_counts(normalize=True); g['x'].idxmax()
g[['a', 'b']].corr(); a.corr(b, method='spearman'); a.cov(b)

# Преобразование и отбор
g['x'].transform('mean'); g['x'].rank(); g['x'].cumsum(); g['x'].diff(); g['x'].pct_change()
g.filter(lambda d: len(d) > 10); g[['a', 'b']].apply(func)      # ключ группы — d.name

# Окна и время
s.rolling(12, min_periods=1).mean(); s.expanding().mean(); s.ewm(span=12).mean()
s.resample('YE').sum()

# Интервалы, сводные таблицы
pd.cut(x, bins, labels, include_lowest=True, retbins=True); pd.qcut(x, 4)
pd.crosstab(a, b, normalize='index')
```